In [2]:
"""
Phase 1.1 — Load Sachs and Asia datasets automatically from bnlearn.
Works on Kaggle (no GPU needed, pure CPU + pandas).
"""

import pandas as pd
import numpy as np
import networkx as nx
from io import StringIO
import urllib.request
import gzip
import os

# ============================================================
# SACHS DATASET (continuous, 11 variables)
# ============================================================

SACHS_URL = "https://www.bnlearn.com/book-crc/code/sachs.interventional.txt.gz"

# Ground-truth DAG for Sachs (11 nodes, 17 edges) — standard from bnlearn
SACHS_GROUND_TRUTH_EDGES = [
    ("Raf", "Mek"),
    ("Mek", "Plcg"),
    ("Mek", "PIP2"),
    ("Mek", "PIP3"),
    ("Plcg", "PIP2"),
    ("Plcg", "PIP3"),
    ("PIP3", "PIP2"),
    ("PIP3", "Akt"),
    ("PIP2", "PKC"),
    ("PIP3", "PKC"),
    ("Akt", "PKA"),
    ("PKC", "PKA"),
    ("PKC", "Jnk"),
    ("PKA", "Jnk"),
    ("PKA", "p38"),
    ("PKA", "Erk"),
    ("Jnk", "p38"),
    ("Jnk", "Erk"),
    ("p38", "Erk"),
]

SACHS_VARIABLES = [
    "Raf", "Mek", "Plcg", "PIP2", "PIP3",
    "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"
]


def load_sachs():
    """
    Download Sachs dataset from bnlearn.
    Returns:
        df: DataFrame with 11 columns (variables) + 'INT' (intervention indicator)
        ground_truth: networkx.DiGraph of the true DAG
    """
    print("[Sachs] Downloading from bnlearn...")
    try:
        # Download gzipped file
        with urllib.request.urlopen(SACHS_URL) as response:
            compressed = response.read()
        decompressed = gzip.decompress(compressed).decode("utf-8")
        df = pd.read_csv(StringIO(decompressed), delimiter=" ")
    except Exception as e:
        print(f"[Sachs] Download failed: {e}")
        print("[Sachs] Trying alternative: read directly with pandas...")
        df = pd.read_csv(SACHS_URL, delimiter=" ", compression="gzip")

    # ------------------------------------------------------------
    # FIX: Normalize column names to match SACHS_VARIABLES
    # The bnlearn file uses "P38" (capital P), while our list uses "p38".
    # We rename any column whose lowercase form matches "p38" to "p38".
    # Also strip whitespace from all column names (safety).
    # ------------------------------------------------------------
    df.columns = df.columns.str.strip()
    df = df.rename(columns={col: "p38" for col in df.columns if col.lower() == "p38"})

    # Keep only the 11 variable columns + intervention indicator
    df = df[SACHS_VARIABLES + ["INT"]].copy()
    df = df.dropna()

    # Build ground-truth DAG
    ground_truth = nx.DiGraph()
    ground_truth.add_nodes_from(SACHS_VARIABLES)
    ground_truth.add_edges_from(SACHS_GROUND_TRUTH_EDGES)

    print(f"[Sachs] Loaded: {df.shape[0]} rows, {len(SACHS_VARIABLES)} variables")
    print(f"[Sachs] Ground truth: {ground_truth.number_of_nodes()} nodes, "
          f"{ground_truth.number_of_edges()} edges")

    return df, ground_truth


# ============================================================
# ASIA DATASET (discrete, 8 variables) — LiM adaptation needed
# ============================================================

# Ground-truth DAG for Asia (8 nodes, 8 edges) — standard from bnlearn
ASIA_GROUND_TRUTH_EDGES = [
    ("asia", "tub"),
    ("smoke", "lung"),
    ("smoke", "bronc"),
    ("tub", "either"),
    ("lung", "either"),
    ("either", "xray"),
    ("either", "dysp"),
    ("bronc", "dysp"),
]

ASIA_VARIABLES = ["asia", "smoke", "tub", "lung", "bronc", "either", "xray", "dysp"]


def load_asia():
    """
    Generate Asia dataset by sampling from the bnlearn Asia Bayesian network.
    Since bnlearn doesn't provide a direct CSV, we use the known CPTs
    to sample data — this is the standard approach.

    Returns:
        df: DataFrame with 8 binary columns
        ground_truth: networkx.DiGraph of the true DAG
    """
    print("[Asia] Generating data from known CPTs...")

    # Known CPTs for Asia (from bnlearn / Lauritzen & Spiegelhalter 1988)
    # We sample in topological order
    np.random.seed(42)
    n_samples = 5000

    # Topological order: asia, smoke, tub, lung, bronc, either, xray, dysp
    asia = np.random.binomial(1, 0.01, n_samples)        # P(asia=1) = 0.01
    smoke = np.random.binomial(1, 0.5, n_samples)        # P(smoke=1) = 0.5

    # tub | asia
    tub = np.where(asia == 1,
                   np.random.binomial(1, 0.05, n_samples),
                   np.random.binomial(1, 0.01, n_samples))

    # lung | smoke
    lung = np.where(smoke == 1,
                    np.random.binomial(1, 0.10, n_samples),
                    np.random.binomial(1, 0.01, n_samples))

    # bronc | smoke
    bronc = np.where(smoke == 1,
                     np.random.binomial(1, 0.60, n_samples),
                     np.random.binomial(1, 0.30, n_samples))

    # either = tub OR lung (deterministic)
    either = np.logical_or(tub, lung).astype(int)

    # xray | either
    xray = np.where(either == 1,
                    np.random.binomial(1, 0.98, n_samples),
                    np.random.binomial(1, 0.05, n_samples))

    # dysp | either, bronc
    p_dysp = np.where(
        (either == 1) & (bronc == 1), 0.90,
        np.where((either == 1) & (bronc == 0), 0.70,
                 np.where((either == 0) & (bronc == 1), 0.40, 0.10)))
    dysp = np.random.binomial(1, p_dysp, n_samples)

    df = pd.DataFrame({
        "asia": asia, "smoke": smoke, "tub": tub, "lung": lung,
        "bronc": bronc, "either": either, "xray": xray, "dysp": dysp
    })

    # Build ground-truth DAG
    ground_truth = nx.DiGraph()
    ground_truth.add_nodes_from(ASIA_VARIABLES)
    ground_truth.add_edges_from(ASIA_GROUND_TRUTH_EDGES)

    print(f"[Asia] Generated: {df.shape[0]} rows, {len(ASIA_VARIABLES)} variables")
    print(f"[Asia] Ground truth: {ground_truth.number_of_nodes()} nodes, "
          f"{ground_truth.number_of_edges()} edges")

    return df, ground_truth


# ============================================================
# QUICK TEST
# ============================================================

if __name__ == "__main__":
    sachs_df, sachs_gt = load_sachs()
    
    print("\n=== Sachs head ===")
    print(sachs_df.head())
    
    # Save for later phases
    os.makedirs("data", exist_ok=True)
    sachs_df.to_csv("data/sachs.csv", index=False)

    asia_df, asia_gt = load_asia()

    print("\n=== Asia head ===")
    print(asia_df.head())
    
    asia_df.to_csv("data/asia.csv", index=False)
    print("\nSaved to data/sachs.csv and data/asia.csv")




[Sachs] Downloading from bnlearn...
[Sachs] Loaded: 5400 rows, 11 variables
[Sachs] Ground truth: 11 nodes, 19 edges

=== Sachs head ===
   Raf  Mek  Plcg  PIP2  PIP3  Erk  Akt  PKA  PKC  Jnk  p38  INT
0    1    1     1     2     3    2    1    3    1    1    2    8
1    1    1     1     1     3    3    2    3    1    1    2    8
2    1    1     2     2     3    2    1    3    2    1    1    8
3    1    1     1     1     3    2    1    3    1    1    3    8
4    1    1     1     1     3    2    1    3    1    1    1    8
[Asia] Generating data from known CPTs...
[Asia] Generated: 5000 rows, 8 variables
[Asia] Ground truth: 8 nodes, 8 edges

=== Asia head ===
   asia  smoke  tub  lung  bronc  either  xray  dysp
0     0      0    0     0      0       0     0     0
1     0      0    0     0      0       0     0     0
2     0      1    0     0      0       0     1     0
3     0      0    0     0      0       0     0     0
4     0      1    0     0      1       0     0     0

Saved to data/

# split_clients.py

In [2]:
"""
Phase 1.2 — Split Sachs/Asia into K clients + inject missingness.
"""

import pandas as pd
import numpy as np


def split_horizontal(df, n_clients, seed=42):
    """
    Split a single dataset into K clients (horizontal federation).
    Each client gets a random subset of rows (same variables).
    """
    rng = np.random.RandomState(seed)
    indices = rng.permutation(len(df))
    splits = np.array_split(indices, n_clients)

    clients = [df.iloc[split].reset_index(drop=True) for split in splits]
    return clients


def split_vertical(df, client_variable_sets):
    """
    Split a single dataset into K clients (vertical federation).
    Each client gets a subset of variables (same rows).
    client_variable_sets: list of lists, e.g. [["Raf","Mek"], ["Mek","Plcg"], ...]
    """
    clients = []
    for varset in client_variable_sets:
        clients.append(df[varset].copy())
    return clients


def inject_mcar(df, missing_rate=0.2, seed=42):
    """MCAR: each cell missing with probability `missing_rate`."""
    rng = np.random.RandomState(seed)
    mask = rng.rand(*df.shape) < missing_rate
    df_missing = df.copy()
    df_missing[mask] = np.nan
    return df_missing


def inject_mar(df, target_col, parent_cols, missing_rate=0.3, seed=42):
    """
    MAR: missingness of target_col depends on observed parent_cols.
    P(R=1 | parents) = sigmoid(standardized parents @ beta), scaled to target rate.
    """
    rng = np.random.RandomState(seed)
    df_missing = df.copy()

    X = df[parent_cols].values.astype(float)
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

    beta = rng.randn(len(parent_cols)) * 0.5
    logits = X @ beta
    # Shift so that the average missing rate matches target
    logits = logits - np.percentile(logits, 100 * (1 - missing_rate))
    probs = 1 / (1 + np.exp(-logits))

    missing_mask = rng.rand(len(df)) < probs
    df_missing.loc[missing_mask, target_col] = np.nan
    return df_missing


def inject_mnar_indirect(df, target_col, mediator_cols, missing_rate=0.3, seed=42):
    """
    MNAR (indirect self-masking): missingness of target_col depends on
    mediator_cols, which are correlated with target_col.
    Allowed by A2 (no direct self-masking).
    """
    return inject_mar(df, target_col, mediator_cols, missing_rate, seed)


def prepare_federated_clients(df, n_clients=3, missing_specs=None, seed=42):
    """
    Full pipeline: split → inject missingness per client.

    Args:
        df: single DataFrame
        n_clients: number of clients
        missing_specs: dict mapping client index → list of missingness specs
            e.g. {0: [{"type": "mcar", "rate": 0.10}],
                  1: [{"type": "mar", "target": "Erk", "parents": ["PKA","Jnk"], "rate": 0.30}]}
        seed: random seed

    Returns:
        clients: list of DataFrames (with NaN for missingness)
        masks: list of dicts (missingness masks per client)
    """
    clients = split_horizontal(df, n_clients, seed=seed)

    if missing_specs is None:
        # Default: each client gets 10% MCAR
        missing_specs = {k: [{"type": "mcar", "rate": 0.10}] for k in range(n_clients)}

    clients_missing = []
    masks = []

    for k, client_df in enumerate(clients):
        specs = missing_specs.get(k, [])
        client_missing = client_df.copy()
        client_masks = {}

        for spec in specs:
            t = spec["type"]
            if t == "mcar":
                client_missing = inject_mcar(client_missing, spec["rate"], seed=seed + k)
                client_masks[f"mcar_{k}"] = client_missing.isna()
            elif t == "mar":
                client_missing = inject_mar(
                    client_missing, spec["target"], spec["parents"],
                    spec["rate"], seed=seed + k)
                client_masks[f"mar_{spec['target']}_{k}"] = client_missing.isna()
            elif t == "mnar":
                client_missing = inject_mnar_indirect(
                    client_missing, spec["target"], spec["mediators"],
                    spec["rate"], seed=seed + k)
                client_masks[f"mnar_{spec['target']}_{k}"] = client_missing.isna()

        clients_missing.append(client_missing)
        masks.append(client_masks)

    return clients_missing, masks


# ============================================================
# QUICK TEST
# ============================================================

if __name__ == "__main__":
    sachs = pd.read_csv("data/sachs.csv")

    # Example: 3 clients, different missingness per client
    missing_specs = {
        0: [{"type": "mcar", "rate": 0.10}],
        1: [{"type": "mar", "target": "Erk", "parents": ["PKA", "Jnk", "p38"], "rate": 0.30}],
        2: [{"type": "mcar", "rate": 0.05},
            {"type": "mar", "target": "Akt", "parents": ["PIP3"], "rate": 0.20}],
    }

    clients, masks = prepare_federated_clients(
        sachs, n_clients=3, missing_specs=missing_specs, seed=42)

    print("=== Federated clients ===")
    for k, c in enumerate(clients):
        print(f"Client {k}: {c.shape[0]} rows, "
              f"{c.isna().sum().sum()} missing cells "
              f"({c.isna().mean().mean():.1%} missing rate)")

=== Federated clients ===
Client 0: 1800 rows, 2190 missing cells (10.1% missing rate)
Client 1: 1800 rows, 664 missing cells (3.1% missing rate)
Client 2: 1800 rows, 1043 missing cells (4.8% missing rate)


# 5 clients split_clients.py

In [3]:
"""
Phase 1.2 — Split Sachs into 5 clients + inject missingness.
"""

import pandas as pd
import numpy as np


def split_horizontal(df, n_clients, seed=42):
    """Split a single dataset into K clients (horizontal federation)."""
    rng = np.random.RandomState(seed)
    indices = rng.permutation(len(df))
    splits = np.array_split(indices, n_clients)
    clients = [df.iloc[split].reset_index(drop=True) for split in splits]
    return clients


def inject_mcar(df, missing_rate=0.2, seed=42):
    """MCAR: each cell missing with probability `missing_rate`."""
    rng = np.random.RandomState(seed)
    mask = rng.rand(*df.shape) < missing_rate
    df_missing = df.copy()
    df_missing[mask] = np.nan
    return df_missing


def inject_mar(df, target_col, parent_cols, missing_rate=0.3, seed=42):
    """
    MAR: missingness of target_col depends on observed parent_cols.
    P(R=1 | parents) = sigmoid(standardized parents @ beta), scaled to target rate.
    """
    rng = np.random.RandomState(seed)
    df_missing = df.copy()

    X = df[parent_cols].values.astype(float)
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

    beta = rng.randn(len(parent_cols)) * 0.5
    logits = X @ beta
    logits = logits - np.percentile(logits, 100 * (1 - missing_rate))
    probs = 1 / (1 + np.exp(-logits))

    missing_mask = rng.rand(len(df)) < probs
    df_missing.loc[missing_mask, target_col] = np.nan
    return df_missing


def inject_mnar_indirect(df, target_col, mediator_cols, missing_rate=0.3, seed=42):
    """MNAR (indirect self-masking): missingness depends on mediators correlated with target."""
    return inject_mar(df, target_col, mediator_cols, missing_rate, seed)


def prepare_federated_clients(df, n_clients=5, missing_specs=None, seed=42):
    """
    Full pipeline: split → inject missingness per client.

    Returns:
        clients: list of DataFrames (with NaN for missingness)
        masks: list of dicts (missingness masks per client)
    """
    clients = split_horizontal(df, n_clients, seed=seed)

    if missing_specs is None:
        missing_specs = {k: [{"type": "mcar", "rate": 0.10}] for k in range(n_clients)}

    clients_missing = []
    masks = []

    for k, client_df in enumerate(clients):
        specs = missing_specs.get(k, [])
        client_missing = client_df.copy()
        client_masks = {}

        for spec in specs:
            t = spec["type"]
            if t == "mcar":
                client_missing = inject_mcar(client_missing, spec["rate"], seed=seed + k)
                client_masks[f"mcar_{k}"] = client_missing.isna()
            elif t == "mar":
                client_missing = inject_mar(
                    client_missing, spec["target"], spec["parents"],
                    spec["rate"], seed=seed + k)
                client_masks[f"mar_{spec['target']}_{k}"] = client_missing.isna()
            elif t == "mnar":
                client_missing = inject_mnar_indirect(
                    client_missing, spec["target"], spec["mediators"],
                    spec["rate"], seed=seed + k)
                client_masks[f"mnar_{spec['target']}_{k}"] = client_missing.isna()

        clients_missing.append(client_missing)
        masks.append(client_masks)

    return clients_missing, masks


# ============================================================
# QUICK TEST — 5 CLIENTS
# ============================================================

if __name__ == "__main__":
    sachs = pd.read_csv("data/sachs.csv")

    # 5 clients, each with a DIFFERENT missingness pattern
    missing_specs = {
        0: [{"type": "mcar", "rate": 0.05}],  # light MCAR
        1: [{"type": "mar", "target": "Erk", "parents": ["PKA", "Jnk", "p38"], "rate": 0.30}],
        2: [{"type": "mar", "target": "Akt", "parents": ["PIP3"], "rate": 0.25}],
        3: [{"type": "mcar", "rate": 0.15}],  # heavier MCAR
        4: [{"type": "mar", "target": "p38", "parents": ["PKA", "Jnk"], "rate": 0.20},
            {"type": "mcar", "rate": 0.05}],
    }

    clients, masks = prepare_federated_clients(
        sachs, n_clients=5, missing_specs=missing_specs, seed=42)

    print("=== 5 Federated Clients ===")
    print(f"Total rows: {sum(len(c) for c in clients)} (should be {len(sachs)})\n")

    for k, c in enumerate(clients):
        missing_cells = c.isna().sum().sum()
        missing_rate = c.isna().mean().mean()
        print(f"Client {k}: {c.shape[0]} rows, "
              f"{missing_cells} missing cells ({missing_rate:.1%} missing)")

    # Save each client
    import os
    os.makedirs("data/clients", exist_ok=True)
    for k, c in enumerate(clients):
        c.to_csv(f"data/clients/client_{k}.csv", index=False)
    print(f"\nSaved 5 clients to data/clients/")

=== 5 Federated Clients ===
Total rows: 5400 (should be 5400)

Client 0: 1080 rows, 635 missing cells (4.9% missing)
Client 1: 1080 rows, 411 missing cells (3.2% missing)
Client 2: 1080 rows, 507 missing cells (3.9% missing)
Client 3: 1080 rows, 1971 missing cells (15.2% missing)
Client 4: 1080 rows, 1102 missing cells (8.5% missing)

Saved 5 clients to data/clients/


# checking the missingness for 5 client 

In [4]:
total = sum(len(pd.read_csv(f"data/clients/client_{k}.csv")) for k in range(5))
print(total)  # should be 5400

5400


In [5]:
for k in range(5):
    c = pd.read_csv(f"data/clients/client_{k}.csv")
    print(f"Client {k}: {c.isna().mean().mean():.1%} missing")

Client 0: 4.9% missing
Client 1: 3.2% missing
Client 2: 3.9% missing
Client 3: 15.2% missing
Client 4: 8.5% missing


# no column is fully missing 

In [6]:
for k in range(5):
    c = pd.read_csv(f"data/clients/client_{k}.csv")
    fully_missing = c.columns[c.isna().all()].tolist()
    if fully_missing:
        print(f"Client {k}: FULLY MISSING columns: {fully_missing}")

In [7]:
# Client 1: Erk should be ~30% missing
c1 = pd.read_csv("data/clients/client_1.csv")
print(f"Client 1 Erk missing: {c1['Erk'].isna().mean():.1%}")  # ~30%

# Client 2: Akt should be ~25% missing
c2 = pd.read_csv("data/clients/client_2.csv")
print(f"Client 2 Akt missing: {c2['Akt'].isna().mean():.1%}")  # ~25%

Client 1 Erk missing: 38.1%
Client 2 Akt missing: 46.9%


# classify_variables.py

In [8]:
"""
Phase 2.1 — Classify variables per client into V_o (fully observed)
and V_m (partially observed), and create R indicators.
"""

import pandas as pd
import numpy as np
import json
import os


def classify_client(df, client_id, missing_threshold=0.05):
    """
    Classify columns into V_o (fully observed) and V_m (partially observed).

    Args:
        df: DataFrame for one client (may contain NaN)
        client_id: int
        missing_threshold: if missing rate < threshold, treat as V_o

    Returns:
        classification: dict with V_o, V_m, R_indicators, missing_rates
    """
    missing_rates = df.isna().mean()

    V_o = []   # fully observed (missing rate < threshold)
    V_m = []   # partially observed (missing rate >= threshold)

    for col in df.columns:
        if col == "INT":  # skip intervention indicator (not a causal variable)
            continue
        if missing_rates[col] < missing_threshold:
            V_o.append(col)
        else:
            V_m.append(col)

    # Create R indicators for each V_m column
    R_indicators = {}
    for col in V_m:
        R_indicators[col] = df[col].isna().astype(int).tolist()

    classification = {
        "client": client_id,
        "V_o": V_o,
        "V_m": V_m,
        "R_indicators": R_indicators,
        "missing_rates": {col: float(missing_rates[col]) for col in df.columns if col != "INT"},
        "n_rows": len(df),
    }

    return classification


def classify_all_clients(client_paths, output_dir="data/phase2"):
    """Classify all clients and save to JSON."""
    os.makedirs(output_dir, exist_ok=True)
    classifications = []

    for k, path in enumerate(client_paths):
        df = pd.read_csv(path)
        cls = classify_client(df, client_id=k)
        classifications.append(cls)

        # Save
        with open(f"{output_dir}/classification_{k}.json", "w") as f:
            json.dump(cls, f, indent=2)

        print(f"Client {k}: V_o={len(cls['V_o'])}, V_m={len(cls['V_m'])}, "
              f"rows={cls['n_rows']}")
        if cls["V_m"]:
            for col in cls["V_m"]:
                print(f"  - {col}: {cls['missing_rates'][col]:.1%} missing")

    return classifications


if __name__ == "__main__":
    client_paths = [f"data/clients/client_{k}.csv" for k in range(5)]
    classifications = classify_all_clients(client_paths)
    print(f"\nSaved classifications to data/phase2/")

Client 0: V_o=7, V_m=4, rows=1080
  - Mek: 5.6% missing
  - Plcg: 6.0% missing
  - PIP3: 5.1% missing
  - PKA: 5.8% missing
Client 1: V_o=10, V_m=1, rows=1080
  - Erk: 38.1% missing
Client 2: V_o=10, V_m=1, rows=1080
  - Akt: 46.9% missing
Client 3: V_o=0, V_m=11, rows=1080
  - Raf: 15.3% missing
  - Mek: 15.6% missing
  - Plcg: 15.6% missing
  - PIP2: 15.1% missing
  - PIP3: 15.9% missing
  - Erk: 12.3% missing
  - Akt: 16.5% missing
  - PKA: 15.7% missing
  - PKC: 16.2% missing
  - Jnk: 15.6% missing
  - p38: 15.1% missing
Client 4: V_o=2, V_m=9, rows=1080
  - Raf: 5.3% missing
  - Mek: 5.6% missing
  - Plcg: 5.6% missing
  - PIP3: 5.4% missing
  - Erk: 5.3% missing
  - Akt: 5.4% missing
  - PKC: 5.0% missing
  - Jnk: 5.2% missing
  - p38: 44.4% missing

Saved classifications to data/phase2/


# missingness_model.py

In [15]:

import json
import pickle
import sys
import traceback

# Scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def log(msg):
    """Print with flush so Kaggle shows output immediately."""
    print(msg, flush=True)

def fit_missingness_model(df, target_col, parent_cols, client_id):
    """
    Fit logistic regression: P(R_target = 1 | parent_cols).

    Handles:
    - Empty parent_cols (fallback to all other columns)
    - NaN in parents (restrict to rows where all parents observed)
    - Insufficient missingness (skip if < 5% or < 10 missing)
    - Insufficient observed rows (skip if < 50)

    Returns:
        dict with model, scaler, coefficients, AUC, metadata
        or None if skipped
    """
    log(f"    [Start] target={target_col}, parents={len(parent_cols)}")

    # Create R indicator (1 = missing, 0 = observed)
    R = df[target_col].isna().astype(int).values
    missing_rate = R.mean()
    log(f"    [Check] missing_rate={missing_rate:.2%}, n_missing={R.sum()}")

    # --- Skip conditions ---
    if missing_rate < 0.05:
        log(f"    [Skip] {target_col}: missing rate {missing_rate:.2%} < 5%")
        return None
    if R.sum() < 10:
        log(f"    [Skip] {target_col}: only {R.sum()} missing (< 10)")
        return None
    if R.sum() == len(R):
        log(f"    [Skip] {target_col}: all values missing")
        return None

    # --- Fallback: if parent_cols is empty, use all other columns ---
    if len(parent_cols) == 0:
        fallback = [c for c in df.columns if c != target_col and c != "INT"]
        log(f"    [Fallback] V_o empty — using {len(fallback)} other columns as parents")
        parent_cols = fallback

    # --- Restrict to rows where all parents are observed ---
    parent_data = df[parent_cols]
    observed_mask = parent_data.notna().all(axis=1).values
    log(f"    [Filter] {observed_mask.sum()} / {len(observed_mask)} rows with all parents observed")

    if observed_mask.sum() < 50:
        log(f"    [Skip] {target_col}: only {observed_mask.sum()} rows with observed parents")
        return None

    X = parent_data.values[observed_mask].astype(float)
    R_obs = R[observed_mask]

    if R_obs.sum() < 10 or R_obs.sum() == len(R_obs):
        log(f"    [Skip] {target_col}: after filtering, {R_obs.sum()} missing / {len(R_obs)} total")
        return None

    # --- Standardize ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # --- Fit logistic regression (L2 regularization) ---
    model = LogisticRegression(
        penalty="l2", C=1.0, max_iter=1000, random_state=42, solver="lbfgs"
    )
    model.fit(X_scaled, R_obs)

    # --- Compute AUC ---
    probs = model.predict_proba(X_scaled)[:, 1]
    auc = roc_auc_score(R_obs, probs)

    # --- Coefficients (on standardized scale) ---
    coefficients = dict(zip(parent_cols, model.coef_[0].tolist()))
    intercept = float(model.intercept_[0])

    top_parent = max(coefficients, key=lambda k: abs(coefficients[k]))
    log(f"    [Fit] {target_col}: AUC={auc:.3f}, top={top_parent} ({coefficients[top_parent]:.3f})")

    return {
        "client": client_id,
        "target": target_col,
        "parents": parent_cols,
        "missing_rate": float(missing_rate),
        "n_missing": int(R.sum()),
        "n_total": len(R),
        "n_used_for_fit": int(observed_mask.sum()),
        "auc": float(auc),
        "coefficients": coefficients,
        "intercept": intercept,
        "model": model,
        "scaler": scaler,
    }
    
def fit_all_models(client_paths, classification_dir="data/phase2", output_dir="data/phase2"):
    os.makedirs(output_dir, exist_ok=True)
    for k, path in enumerate(client_paths):
        log(f"\n{'='*60}")
        log(f"Client {k}: loading {path}")
        log(f"{'='*60}")
        df = pd.read_csv(path)
        log(f"  Data shape: {df.shape}")

        cls_path = f"{classification_dir}/classification_{k}.json"
        if not os.path.exists(cls_path):
            log(f"  [ERROR] Classification file not found: {cls_path}")
            continue

        with open(cls_path) as f:
            cls = json.load(f)

        V_o = cls["V_o"]
        V_m = cls["V_m"]
        log(f"  V_o ({len(V_o)}): {V_o}")
        log(f"  V_m ({len(V_m)}): {V_m}")

        if not V_m:
            log(f"  [Skip] No V_m columns")
            continue

        for target_col in V_m:
            try:
                model_dict = fit_missingness_model(df, target_col, V_o, client_id=k)
                if model_dict is None:
                    continue

                # Save model
                with open(f"{output_dir}/missingness_model_{k}_{target_col}.pkl", "wb") as f:
                    pickle.dump({"model": model_dict["model"],
                                 "scaler": model_dict["scaler"]}, f)

                # Save metadata
                meta = {kk: vv for kk, vv in model_dict.items()
                        if kk not in ["model", "scaler"]}
                with open(f"{output_dir}/missingness_meta_{k}_{target_col}.json", "w") as f:
                    json.dump(meta, f, indent=2)

                log(f"    [Saved] model + meta for {target_col}")

            except Exception as e:
                log(f"    [ERROR] {target_col}: {type(e).__name__}: {e}")
                import traceback
                traceback.print_exc()


if __name__ == "__main__":
    client_paths = [f"data/clients/client_{k}.csv" for k in range(5)]
    fit_all_models(client_paths)

    # Summary: count saved models
    import os
    meta_files = [f for f in os.listdir("data/phase2") if f.startswith("missingness_meta_")]
    pkl_files = [f for f in os.listdir("data/phase2") if f.startswith("missingness_model_") and f.endswith(".pkl")]
    print(f"\n{'='*60}")
    print(f"SUMMARY")
    print(f"{'='*60}")
    print(f"  Meta files: {len(meta_files)}")
    print(f"  Model files: {len(pkl_files)}")
    print(f"  Expected: 26 models (Client 0: 4, Client 1: 1, Client 2: 1, Client 3: 11, Client 4: 9)")
    if len(pkl_files) == 26:
        print(f"  ✓ All models saved successfully")
    else:
        print(f"  ⚠️ Expected 26, got {len(pkl_files)} — check logs above for [Skip] or [ERROR]")


Client 0: loading data/clients/client_0.csv
  Data shape: (1080, 12)
  V_o (7): ['Raf', 'PIP2', 'Erk', 'Akt', 'PKC', 'Jnk', 'p38']
  V_m (4): ['Mek', 'Plcg', 'PIP3', 'PKA']
    [Start] target=Mek, parents=7
    [Check] missing_rate=5.65%, n_missing=61
    [Filter] 797 / 1080 rows with all parents observed
    [Fit] Mek: AUC=0.600, top=Erk (-0.281)
    [Saved] model + meta for Mek
    [Start] target=Plcg, parents=7
    [Check] missing_rate=6.02%, n_missing=65
    [Filter] 797 / 1080 rows with all parents observed
    [Fit] Plcg: AUC=0.620, top=p38 (-0.247)
    [Saved] model + meta for Plcg
    [Start] target=PIP3, parents=7
    [Check] missing_rate=5.09%, n_missing=55
    [Filter] 797 / 1080 rows with all parents observed
    [Fit] PIP3: AUC=0.634, top=p38 (-0.564)
    [Saved] model + meta for PIP3
    [Start] target=PKA, parents=7
    [Check] missing_rate=5.83%, n_missing=63
    [Filter] 797 / 1080 rows with all parents observed
    [Fit] PKA: AUC=0.626, top=p38 (-0.268)
    [Saved] m

# only need to run if the above classification does not output

In [17]:
import json
import os

# Check files exist
print("=== Files in data/phase2/ ===")
for f in sorted(os.listdir("data/phase2")):
    print(f"  {f}")

# Check classification for Client 3
print("\n=== Client 3 classification ===")
with open("data/phase2/classification_3.json") as f:
    cls = json.load(f)
print(f"V_o: {cls['V_o']}")
print(f"V_m: {cls['V_m']}")
print(f"n_rows: {cls['n_rows']}")

# Check client 3 data
import pandas as pd
c3 = pd.read_csv("data/clients/client_3.csv")
print(f"\n=== Client 3 data ===")
print(f"Shape: {c3.shape}")
print(f"Missing rates:\n{c3.isna().mean()}")

=== Files in data/phase2/ ===
  classification_0.json
  classification_1.json
  classification_2.json
  classification_3.json
  classification_4.json
  missingness_meta_0_Mek.json
  missingness_meta_0_PIP3.json
  missingness_meta_0_PKA.json
  missingness_meta_0_Plcg.json
  missingness_meta_1_Erk.json
  missingness_meta_2_Akt.json
  missingness_model_0_Mek.pkl
  missingness_model_0_PIP3.pkl
  missingness_model_0_PKA.pkl
  missingness_model_0_Plcg.pkl
  missingness_model_1_Erk.pkl
  missingness_model_2_Akt.pkl

=== Client 3 classification ===
V_o: []
V_m: ['Raf', 'Mek', 'Plcg', 'PIP2', 'PIP3', 'Erk', 'Akt', 'PKA', 'PKC', 'Jnk', 'p38']
n_rows: 1080

=== Client 3 data ===
Shape: (1080, 12)
Missing rates:
Raf     0.152778
Mek     0.155556
Plcg    0.156481
PIP2    0.150926
PIP3    0.159259
Erk     0.123148
Akt     0.164815
PKA     0.157407
PKC     0.162037
Jnk     0.155556
p38     0.150926
INT     0.136111
dtype: float64


# Phase 3A — IPW Weights + Corrected Data

In [16]:
"""
Phase 3A — Compute IPW weights and apply correction per client.
"""

def log(msg):
    print(msg, flush=True)


def compute_weights_for_column(df, target_col, model_dict):
    """
    Compute IPW weights for one V_m column.

    w_i = 1 / P(R_i = 0 | parents)

    Returns:
        weights: np.array of shape (n_rows,) — weight per row (1.0 for rows
                 where target is missing, since they don't contribute to
                 the observed-data distribution)
        observed_mask: np.array of bool — which rows have target observed
    """
    model = model_dict["model"]
    scaler = model_dict["scaler"]
    parents = model_dict["parents"]

    # Observed mask for target
    observed_mask = df[target_col].notna().values

    # For rows where target is observed, compute P(R = 0 | parents)
    # For rows where target is missing, weight = 1.0 (they're excluded anyway)
    weights = np.ones(len(df))

    if observed_mask.sum() == 0:
        return weights, observed_mask

    # Get parent values for observed rows
    parent_data = df.loc[observed_mask, parents]

    # Check for NaN in parents (shouldn't happen if parents are V_o, but safety)
    if parent_data.isna().any().any():
        log(f"    [Warn] {target_col}: NaN in parents for observed rows — using mean imputation")
        parent_data = parent_data.fillna(parent_data.mean())

    X = parent_data.values.astype(float)
    X_scaled = scaler.transform(X)

    # P(R = 1 | parents) — probability of being missing
    p_missing = model.predict_proba(X_scaled)[:, 1]

    # Clip to avoid division by zero
    p_missing = np.clip(p_missing, 1e-6, 1 - 1e-6)

    # P(R = 0 | parents) = 1 - p_missing
    p_observed = 1 - p_missing

    # Weight = 1 / P(observed)
    w = 1.0 / p_observed

    # Clip weights at 95th percentile to avoid extreme values
    w_clip = np.percentile(w, 95)
    w = np.minimum(w, w_clip)

    weights[observed_mask] = w

    return weights, observed_mask


def compute_client_weights(client_df, classification, models_dir="data/phase2",
                           output_dir="data/phase3"):
    """
    Compute combined IPW weights for one client (across all V_m columns).

    The combined weight for row i is the product of weights across all V_m columns:
        w_i = prod_j w_{i,j}

    This is because each V_m column contributes its own selection bias.
    """
    os.makedirs(output_dir, exist_ok=True)
    client_id = classification["client"]
    V_m = classification["V_m"]

    n_rows = len(client_df)
    combined_weights = np.ones(n_rows)

    per_column_info = {}

    for target_col in V_m:
        model_path = f"{models_dir}/missingness_model_{client_id}_{target_col}.pkl"
        if not os.path.exists(model_path):
            log(f"    [Skip] {target_col}: model not found at {model_path}")
            continue

        with open(model_path, "rb") as f:
            model_dict = pickle.load(f)

        # Reconstruct model_dict with metadata
        meta_path = f"{models_dir}/missingness_meta_{client_id}_{target_col}.json"
        with open(meta_path) as f:
            meta = json.load(f)

        model_dict.update(meta)  # add parents, target, etc.

        weights, observed_mask = compute_weights_for_column(client_df, target_col, model_dict)

        combined_weights *= weights

        per_column_info[target_col] = {
            "n_observed": int(observed_mask.sum()),
            "n_missing": int((~observed_mask).sum()),
            "weight_mean": float(weights[observed_mask].mean()) if observed_mask.sum() > 0 else 1.0,
            "weight_max": float(weights[observed_mask].max()) if observed_mask.sum() > 0 else 1.0,
        }

        log(f"    [Weights] {target_col}: n_obs={observed_mask.sum()}, "
            f"mean_w={per_column_info[target_col]['weight_mean']:.3f}, "
            f"max_w={per_column_info[target_col]['weight_max']:.3f}")

    # Normalize weights to have mean 1 (so effective sample size is preserved)
    combined_weights = combined_weights / combined_weights.mean()

    # Effective sample size
    n_eff = (combined_weights.sum() ** 2) / (combined_weights ** 2).sum()

    log(f"  Combined: n_rows={n_rows}, n_eff={n_eff:.1f} "
        f"({n_eff/n_rows:.1%} of original)")

    return {
        "client": client_id,
        "combined_weights": combined_weights,
        "per_column_info": per_column_info,
        "n_eff": float(n_eff),
        "n_rows": n_rows,
    }


def correct_all_clients(client_paths, classification_dir="data/phase2",
                        models_dir="data/phase2", output_dir="data/phase3"):
    """Compute weights and save corrected data for all clients."""
    os.makedirs(output_dir, exist_ok=True)

    all_results = []

    for k, path in enumerate(client_paths):
        log(f"\n{'='*60}")
        log(f"Client {k}: computing IPW weights")
        log(f"{'='*60}")

        df = pd.read_csv(path)

        with open(f"{classification_dir}/classification_{k}.json") as f:
            cls = json.load(f)

        result = compute_client_weights(df, cls, models_dir=models_dir, output_dir=output_dir)

        # Save weights
        weights_df = pd.DataFrame({
            "row": np.arange(len(df)),
            "weight": result["combined_weights"],
        })
        weights_df.to_csv(f"{output_dir}/weights_{k}.csv", index=False)

        # Save corrected data (data unchanged; weights stored separately)
        # For cumulants, we'll use the weights directly — no need to modify data
        all_results.append(result)

        # Save metadata
        meta = {
            "client": k,
            "n_rows": result["n_rows"],
            "n_eff": result["n_eff"],
            "per_column_info": result["per_column_info"],
        }
        with open(f"{output_dir}/ipw_meta_{k}.json", "w") as f:
            json.dump(meta, f, indent=2)

    log(f"\n{'='*60}")
    log("Phase 3A complete.")
    log(f"{'='*60}")

    return all_results


if __name__ == "__main__":
    client_paths = [f"data/clients/client_{k}.csv" for k in range(5)]
    correct_all_clients(client_paths)

    # Summary
    log(f"\n{'='*60}")
    log("SUMMARY")
    log(f"{'='*60}")
    for k in range(5):
        meta_path = f"data/phase3/ipw_meta_{k}.json"
        if os.path.exists(meta_path):
            with open(meta_path) as f:
                meta = json.load(f)
            log(f"  Client {k}: n_eff={meta['n_eff']:.1f} / {meta['n_rows']} "
                f"({meta['n_eff']/meta['n_rows']:.1%})")


Client 0: computing IPW weights
    [Warn] Mek: NaN in parents for observed rows — using mean imputation
    [Weights] Mek: n_obs=1019, mean_w=1.060, max_w=1.096
    [Warn] Plcg: NaN in parents for observed rows — using mean imputation
    [Weights] Plcg: n_obs=1015, mean_w=1.070, max_w=1.122
    [Warn] PIP3: NaN in parents for observed rows — using mean imputation
    [Weights] PIP3: n_obs=1025, mean_w=1.064, max_w=1.118
    [Warn] PKA: NaN in parents for observed rows — using mean imputation
    [Weights] PKA: n_obs=1017, mean_w=1.059, max_w=1.099
  Combined: n_rows=1080, n_eff=1076.7 (99.7% of original)

Client 1: computing IPW weights
    [Weights] Erk: n_obs=669, mean_w=1.603, max_w=2.306
  Combined: n_rows=1080, n_eff=996.9 (92.3% of original)

Client 2: computing IPW weights
    [Weights] Akt: n_obs=573, mean_w=1.875, max_w=2.599
  Combined: n_rows=1080, n_eff=970.5 (89.9% of original)

Client 3: computing IPW weights
    [Warn] Raf: NaN in parents for observed rows — using mea

# 3B — Star Graphs + Weighted Cumulants

## start_graph_cumulants.py

In [17]:
"""
Phase 3B.1 — Decompose each client's skeleton into star graphs.
Uses the ground-truth skeleton (undirected version of the true DAG).
"""

import networkx as nx


def log(msg):
    print(msg, flush=True)


# Ground-truth DAG edges (same as in Phase 1)
SACHS_GROUND_TRUTH_EDGES = [
    ("Raf", "Mek"), ("Mek", "Plcg"), ("Mek", "PIP2"), ("Mek", "PIP3"),
    ("Plcg", "PIP2"), ("Plcg", "PIP3"), ("PIP3", "PIP2"), ("PIP3", "Akt"),
    ("PIP2", "PKC"), ("PIP3", "PKC"), ("Akt", "PKA"), ("PKC", "PKA"),
    ("PKC", "Jnk"), ("PKA", "Jnk"), ("PKA", "p38"), ("PKA", "Erk"),
    ("Jnk", "p38"), ("Jnk", "Erk"), ("p38", "Erk"),
]

SACHS_VARIABLES = [
    "Raf", "Mek", "Plcg", "PIP2", "PIP3",
    "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"
]


def build_ground_truth_skeleton():
    """Build the undirected skeleton from the ground-truth DAG."""
    G = nx.Graph()
    G.add_nodes_from(SACHS_VARIABLES)
    for u, v in SACHS_GROUND_TRUTH_EDGES:
        G.add_edge(u, v)
    return G


def decompose_into_star_graphs(skeleton, variables):
    """
    Decompose a skeleton into N star graphs (one per node).

    Each star graph centered at v_c has:
    - center: v_c
    - peripherals: all neighbors of v_c in the skeleton

    Returns:
        list of dicts: [{"center": v_c, "peripherals": [...]}, ...]
    """
    star_graphs = []
    for center in variables:
        if center not in skeleton:
            continue
        peripherals = list(skeleton.neighbors(center))
        if len(peripherals) == 0:
            log(f"  [Skip] {center}: no neighbors (isolated node)")
            continue
        star_graphs.append({
            "center": center,
            "peripherals": peripherals,
        })
    return star_graphs


def decompose_all_clients(n_clients=5, output_dir="data/phase3b"):
    """Decompose the skeleton into star graphs (same for all clients,
    since they share the same variables — horizontal federation)."""
    os.makedirs(output_dir, exist_ok=True)

    skeleton = build_ground_truth_skeleton()
    log(f"Skeleton: {skeleton.number_of_nodes()} nodes, "
        f"{skeleton.number_of_edges()} edges")

    star_graphs = decompose_into_star_graphs(skeleton, SACHS_VARIABLES)
    log(f"Decomposed into {len(star_graphs)} star graphs")

    for sg in star_graphs:
        log(f"  Center: {sg['center']}, Peripherals: {sg['peripherals']}")

    # Save (same star graphs for all clients, since horizontal federation)
    for k in range(n_clients):
        with open(f"{output_dir}/star_graphs_{k}.json", "w") as f:
            json.dump({
                "client": k,
                "skeleton_edges": list(skeleton.edges()),
                "star_graphs": star_graphs,
            }, f, indent=2)

    log(f"\nSaved star graphs for {n_clients} clients to {output_dir}/")
    return star_graphs


if __name__ == "__main__":
    decompose_all_clients(n_clients=5)

Skeleton: 11 nodes, 19 edges
Decomposed into 11 star graphs
  Center: Raf, Peripherals: ['Mek']
  Center: Mek, Peripherals: ['Raf', 'Plcg', 'PIP2', 'PIP3']
  Center: Plcg, Peripherals: ['Mek', 'PIP2', 'PIP3']
  Center: PIP2, Peripherals: ['Mek', 'Plcg', 'PIP3', 'PKC']
  Center: PIP3, Peripherals: ['Mek', 'Plcg', 'PIP2', 'Akt', 'PKC']
  Center: Erk, Peripherals: ['PKA', 'Jnk', 'p38']
  Center: Akt, Peripherals: ['PIP3', 'PKA']
  Center: PKA, Peripherals: ['Akt', 'PKC', 'Jnk', 'p38', 'Erk']
  Center: PKC, Peripherals: ['PIP2', 'PIP3', 'PKA', 'Jnk']
  Center: Jnk, Peripherals: ['PKC', 'PKA', 'p38', 'Erk']
  Center: p38, Peripherals: ['PKA', 'Jnk', 'Erk']

Saved star graphs for 5 clients to data/phase3b/


# weighted_cumulants.py

In [25]:
"""
Phase 3B.2 — Compute weighted 3rd-order cumulants per star graph.
Uses IPW weights from Phase 3A, restricted to rows where both
variables are observed (with renormalized weights).
"""

import pandas as pd
import numpy as np
import json
import os


def log(msg):
    print(msg, flush=True)


# ============================================================
# WEIGHTED CUMULANT FUNCTIONS
# ============================================================

def weighted_mean(x, w):
    """Weighted mean."""
    return np.sum(w * x) / np.sum(w)


def weighted_c3(x, w):
    """Weighted 3rd-order auto-cumulant of x."""
    mu = weighted_mean(x, w)
    return np.sum(w * (x - mu) ** 3) / np.sum(w)


def weighted_c12(x, y, w):
    """Weighted cross-cumulant C_{1,2}(x, y) = E[(x-mu_x)(y-mu_y)^2]."""
    mu_x = weighted_mean(x, w)
    mu_y = weighted_mean(y, w)
    return np.sum(w * (x - mu_x) * (y - mu_y) ** 2) / np.sum(w)


def weighted_c21(x, y, w):
    """Weighted cross-cumulant C_{2,1}(x, y) = E[(x-mu_x)^2(y-mu_y)]."""
    mu_x = weighted_mean(x, w)
    mu_y = weighted_mean(y, w)
    return np.sum(w * (x - mu_x) ** 2 * (y - mu_y)) / np.sum(w)


# ============================================================
# PAIR CUMULANTS
# ============================================================

def compute_pair_cumulants(df, weights, var_x, var_y, min_obs=50):
    """..."""
    mask = df[var_x].notna() & df[var_y].notna()
    n_obs = mask.sum()
    if n_obs < min_obs:
        return None

    x = df.loc[mask, var_x].values.astype(float)
    y = df.loc[mask, var_y].values.astype(float)
    w = weights[mask]
    w = w * (n_obs / w.sum())

    return {
        "C3_x": float(weighted_c3(x, w)),
        "C3_y": float(weighted_c3(y, w)),
        "C12_xy": float(weighted_c12(x, y, w)),   # C12(x, y)
        "C21_xy": float(weighted_c21(x, y, w)),   # C21(x, y)
        "C12_yx": float(weighted_c12(y, x, w)),   # C12(y, x)  ← NEW
        "C21_yx": float(weighted_c21(y, x, w)),   # C21(y, x)  ← NEW
        "n_obs": int(n_obs),
        "weight_mean": float(w.mean()),
        "weight_max": float(w.max()),
    }


# ============================================================
# STAR GRAPH CUMULANTS
# ============================================================

def compute_star_graph_cumulants(df, weights, star_graph, min_obs=50):
    """..."""
    center = star_graph["center"]
    peripherals = star_graph["peripherals"]

    cumulants = {}
    n_obs_info = {}
    weight_info = {}
    skipped = []

    for p in peripherals:
        pair_cum = compute_pair_cumulants(df, weights, center, p, min_obs=min_obs)
        if pair_cum is None:
            skipped.append(p)
            continue

        # Store center auto-cumulant (only once)
        cumulants[f"C3_{center}"] = pair_cum["C3_x"]
        cumulants[f"C3_{p}"] = pair_cum["C3_y"]

        # Store BOTH directions of cross-cumulants
        cumulants[f"C12_{center}_{p}"] = pair_cum["C12_xy"]
        cumulants[f"C21_{center}_{p}"] = pair_cum["C21_xy"]
        cumulants[f"C12_{p}_{center}"] = pair_cum["C12_yx"]  # ← NEW
        cumulants[f"C21_{p}_{center}"] = pair_cum["C21_yx"]  # ← NEW

        n_obs_info[f"{center}_{p}"] = pair_cum["n_obs"]
        weight_info[f"{center}_{p}"] = {
            "mean": pair_cum["weight_mean"],
            "max": pair_cum["weight_max"],
        }

    return {
        "center": center,
        "peripherals": peripherals,
        "cumulants": cumulants,
        "n_obs_pairs": n_obs_info,
        "weight_info": weight_info,
        "skipped_peripherals": skipped,
    }


# ============================================================
# MAIN PIPELINE
# ============================================================

def compute_all_clients(n_clients=5, data_dir="data",
                        phase3_dir="data/phase3",
                        phase3b_dir="data/phase3b",
                        output_dir="data/phase3b",
                        min_obs=50):
    """Compute weighted cumulants per star graph for all clients."""
    os.makedirs(output_dir, exist_ok=True)

    for k in range(n_clients):
        log(f"\n{'='*60}")
        log(f"Client {k}: computing weighted cumulants")
        log(f"{'='*60}")

        # Load data
        df = pd.read_csv(f"{data_dir}/clients/client_{k}.csv")
        log(f"  Data: {df.shape[0]} rows, {df.shape[1]} cols")

        # Load weights
        weights_df = pd.read_csv(f"{phase3_dir}/weights_{k}.csv")
        weights = weights_df["weight"].values
        log(f"  Weights: mean={weights.mean():.3f}, max={weights.max():.3f}")

        # Load star graphs
        with open(f"{phase3b_dir}/star_graphs_{k}.json") as f:
            sg_data = json.load(f)

        star_graphs = sg_data["star_graphs"]
        log(f"  {len(star_graphs)} star graphs to process")

        # Compute cumulants per star graph
        client_cumulants = {
            "client": k,
            "n_rows": len(df),
            "star_graphs": [],
        }

        total_cumulants = 0
        total_skipped = 0

        for sg in star_graphs:
            sg_cum = compute_star_graph_cumulants(
                df, weights, sg, min_obs=min_obs)
            client_cumulants["star_graphs"].append(sg_cum)

            n_cum = len(sg_cum["cumulants"])
            n_skip = len(sg_cum["skipped_peripherals"])
            total_cumulants += n_cum
            total_skipped += n_skip

            status = "✓" if n_skip == 0 else f"⚠️ {n_skip} skipped"
            log(f"  Star graph (center={sg['center']}, deg={len(sg['peripherals'])}): "
                f"{n_cum} cumulants, {len(sg_cum['n_obs_pairs'])} pairs {status}")

        # Save
        with open(f"{output_dir}/cumulants_{k}.json", "w") as f:
            json.dump(client_cumulants, f, indent=2)

        log(f"  TOTAL: {total_cumulants} cumulants, {total_skipped} skipped pairs")
        log(f"  Saved to {output_dir}/cumulants_{k}.json")

    log(f"\n{'='*60}")
    log("Phase 3B.2 complete.")
    log(f"{'='*60}")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    compute_all_clients(n_clients=5, min_obs=50)

    # Summary
    log(f"\n{'='*60}")
    log("SUMMARY")
    log(f"{'='*60}")

    for k in range(5):
        path = f"data/phase3b/cumulants_{k}.json"
        if not os.path.exists(path):
            log(f"  Client {k}: [MISSING]")
            continue

        with open(path) as f:
            data = json.load(f)

        n_star = len(data["star_graphs"])
        n_cum = sum(len(sg["cumulants"]) for sg in data["star_graphs"])
        n_pairs = sum(len(sg["n_obs_pairs"]) for sg in data["star_graphs"])
        n_skipped = sum(len(sg["skipped_peripherals"]) for sg in data["star_graphs"])

        log(f"  Client {k}: {n_star} star graphs, "
            f"{n_cum} cumulants, {n_pairs} pairs, {n_skipped} skipped")


Client 0: computing weighted cumulants
  Data: 1080 rows, 12 cols
  Weights: mean=1.000, max=1.160
  11 star graphs to process
  Star graph (center=Raf, deg=1): 6 cumulants, 1 pairs ✓
  Star graph (center=Mek, deg=4): 21 cumulants, 4 pairs ✓
  Star graph (center=Plcg, deg=3): 16 cumulants, 3 pairs ✓
  Star graph (center=PIP2, deg=4): 21 cumulants, 4 pairs ✓
  Star graph (center=PIP3, deg=5): 26 cumulants, 5 pairs ✓
  Star graph (center=Erk, deg=3): 16 cumulants, 3 pairs ✓
  Star graph (center=Akt, deg=2): 11 cumulants, 2 pairs ✓
  Star graph (center=PKA, deg=5): 26 cumulants, 5 pairs ✓
  Star graph (center=PKC, deg=4): 21 cumulants, 4 pairs ✓
  Star graph (center=Jnk, deg=4): 21 cumulants, 4 pairs ✓
  Star graph (center=p38, deg=3): 16 cumulants, 3 pairs ✓
  TOTAL: 201 cumulants, 0 skipped pairs
  Saved to data/phase3b/cumulants_0.json

Client 1: computing weighted cumulants
  Data: 1080 rows, 12 cols
  Weights: mean=1.000, max=1.679
  11 star graphs to process
  Star graph (center=Ra

# package_cumulants.py

In [26]:
"""
Phase 3B.3 — Package all client cumulants into a single file per client
for transmission to the server. Includes n_eff for aggregation weighting.
"""

import json
import os


def log(msg):
    print(msg, flush=True)


def package_client(client_id, phase3_dir="data/phase3",
                   phase3b_dir="data/phase3b",
                   output_dir="data/phase3b"):
    """
    Package one client's cumulants + metadata for server transmission.

    Output:
        client_cumulants_{k}.json:
        {
            "client": k,
            "n_rows": int,
            "n_eff": float,
            "n_eff_ratio": float,
            "star_graphs": [...]  # from cumulants_{k}.json
        }
    """
    # Load cumulants
    cum_path = f"{phase3b_dir}/cumulants_{client_id}.json"
    if not os.path.exists(cum_path):
        log(f"  [ERROR] Client {client_id}: cumulants file not found at {cum_path}")
        return None

    with open(cum_path) as f:
        cumulants = json.load(f)

    # Load IPW metadata (for n_eff)
    ipw_path = f"{phase3_dir}/ipw_meta_{client_id}.json"
    if not os.path.exists(ipw_path):
        log(f"  [ERROR] Client {client_id}: IPW meta not found at {ipw_path}")
        return None

    with open(ipw_path) as f:
        ipw_meta = json.load(f)

    # Package
    packaged = {
        "client": client_id,
        "n_rows": cumulants["n_rows"],
        "n_eff": ipw_meta["n_eff"],
        "n_eff_ratio": ipw_meta["n_eff"] / ipw_meta["n_rows"],
        "star_graphs": cumulants["star_graphs"],
    }

    # Save
    out_path = f"{output_dir}/client_cumulants_{client_id}.json"
    with open(out_path, "w") as f:
        json.dump(packaged, f, indent=2)

    # Count cumulants
    n_cumulants = sum(len(sg["cumulants"]) for sg in packaged["star_graphs"])
    n_pairs = sum(len(sg["n_obs_pairs"]) for sg in packaged["star_graphs"])

    log(f"  Client {client_id}: {len(packaged['star_graphs'])} star graphs, "
        f"{n_cumulants} cumulants, {n_pairs} pairs, "
        f"n_eff={packaged['n_eff']:.1f} ({packaged['n_eff_ratio']:.1%})")

    return packaged


def package_all(n_clients=5, phase3_dir="data/phase3",
                phase3b_dir="data/phase3b", output_dir="data/phase3b"):
    """Package all clients."""
    log(f"{'='*60}")
    log("Packaging client cumulants for server transmission")
    log(f"{'='*60}")

    packaged_clients = []
    for k in range(n_clients):
        result = package_client(k, phase3_dir=phase3_dir,
                                phase3b_dir=phase3b_dir,
                                output_dir=output_dir)
        if result is not None:
            packaged_clients.append(result)

    log(f"\n{'='*60}")
    log(f"Packaged {len(packaged_clients)} / {n_clients} clients")
    log(f"{'='*60}")

    return packaged_clients


if __name__ == "__main__":
    packaged = package_all(n_clients=5)

    # Summary
    log(f"\n{'='*60}")
    log("SUMMARY")
    log(f"{'='*60}")

    total_cumulants = 0
    total_pairs = 0
    total_n_eff = 0

    for p in packaged:
        n_cum = sum(len(sg["cumulants"]) for sg in p["star_graphs"])
        n_pairs = sum(len(sg["n_obs_pairs"]) for sg in p["star_graphs"])
        total_cumulants += n_cum
        total_pairs += n_pairs
        total_n_eff += p["n_eff"]

        log(f"  Client {p['client']}: "
            f"{len(p['star_graphs'])} star graphs, "
            f"{n_cum} cumulants, {n_pairs} pairs, "
            f"n_eff={p['n_eff']:.1f} ({p['n_eff_ratio']:.1%})")

    log(f"\n  TOTAL: {total_cumulants} cumulants, {total_pairs} pairs")
    log(f"  Average n_eff: {total_n_eff / len(packaged):.1f} / 1080 "
        f"({total_n_eff / len(packaged) / 1080:.1%})")

    # Check files exist
    log(f"\n  Files saved:")
    for k in range(5):
        path = f"data/phase3b/client_cumulants_{k}.json"
        if os.path.exists(path):
            size_kb = os.path.getsize(path) / 1024
            log(f"    client_cumulants_{k}.json ({size_kb:.1f} KB)")
        else:
            log(f"    [MISSING] client_cumulants_{k}.json")

Packaging client cumulants for server transmission
  Client 0: 11 star graphs, 201 cumulants, 38 pairs, n_eff=1076.7 (99.7%)
  Client 1: 11 star graphs, 201 cumulants, 38 pairs, n_eff=996.9 (92.3%)
  Client 2: 11 star graphs, 201 cumulants, 38 pairs, n_eff=970.5 (89.9%)
  Client 3: 11 star graphs, 201 cumulants, 38 pairs, n_eff=935.7 (86.6%)
  Client 4: 11 star graphs, 201 cumulants, 38 pairs, n_eff=1000.3 (92.6%)

Packaged 5 / 5 clients

SUMMARY
  Client 0: 11 star graphs, 201 cumulants, 38 pairs, n_eff=1076.7 (99.7%)
  Client 1: 11 star graphs, 201 cumulants, 38 pairs, n_eff=996.9 (92.3%)
  Client 2: 11 star graphs, 201 cumulants, 38 pairs, n_eff=970.5 (89.9%)
  Client 3: 11 star graphs, 201 cumulants, 38 pairs, n_eff=935.7 (86.6%)
  Client 4: 11 star graphs, 201 cumulants, 38 pairs, n_eff=1000.3 (92.6%)

  TOTAL: 1005 cumulants, 190 pairs
  Average n_eff: 996.0 / 1080 (92.2%)

  Files saved:
    client_cumulants_0.json (15.8 KB)
    client_cumulants_1.json (15.6 KB)
    client_cumul

# aggregate_cumulants.py

In [27]:
"""
Phase 4.1 — Server aggregates cumulants from all clients using n_eff weights.
"""

import json
import os
import numpy as np


def log(msg):
    print(msg, flush=True)


def load_all_clients(n_clients=5, phase3b_dir="data/phase3b"):
    """Load all client cumulant files."""
    clients = []
    for k in range(n_clients):
        path = f"{phase3b_dir}/client_cumulants_{k}.json"
        if not os.path.exists(path):
            log(f"  [ERROR] Client {k}: file not found at {path}")
            continue
        with open(path) as f:
            clients.append(json.load(f))
    log(f"Loaded {len(clients)} clients")
    return clients


def aggregate_cumulants(clients):
    """
    Aggregate cumulants across clients using n_eff weights.

    For each star graph (by center), for each cumulant key:
        C_global = sum_k (n_eff_k / sum_j n_eff_j) * C_k

    Returns:
        dict: {
            "n_clients": K,
            "total_n_eff": float,
            "star_graphs": [
                {
                    "center": ...,
                    "peripherals": [...],
                    "cumulants": {...},  # aggregated
                    "n_obs_pairs": {...},  # total observed rows
                    "n_clients_contributing": {...},  # how many clients contributed
                }
            ]
        }
    """
    n_clients = len(clients)
    total_n_eff = sum(c["n_eff"] for c in clients)
    log(f"Aggregating {n_clients} clients, total n_eff = {total_n_eff:.1f}")

    # Build index: center → list of (client, star_graph)
    star_graphs_by_center = {}
    for client in clients:
        for sg in client["star_graphs"]:
            center = sg["center"]
            if center not in star_graphs_by_center:
                star_graphs_by_center[center] = []
            star_graphs_by_center[center].append((client, sg))

    # Aggregate per center
    aggregated = []
    for center, sg_list in star_graphs_by_center.items():
        # All star graphs for this center should have the same peripherals
        peripherals = sg_list[0][1]["peripherals"]

        # Aggregate cumulants
        agg_cumulants = {}
        agg_n_obs = {}
        n_contributing = {}

        for client, sg in sg_list:
            weight = client["n_eff"] / total_n_eff
            for key, val in sg["cumulants"].items():
                if key not in agg_cumulants:
                    agg_cumulants[key] = 0.0
                agg_cumulants[key] += weight * val
            for pair_key, n_obs in sg["n_obs_pairs"].items():
                if pair_key not in agg_n_obs:
                    agg_n_obs[pair_key] = 0
                agg_n_obs[pair_key] += n_obs
                if pair_key not in n_contributing:
                    n_contributing[pair_key] = 0
                n_contributing[pair_key] += 1

        aggregated.append({
            "center": center,
            "peripherals": peripherals,
            "cumulants": agg_cumulants,
            "n_obs_pairs": agg_n_obs,
            "n_clients_contributing": n_contributing,
        })

    return {
        "n_clients": n_clients,
        "total_n_eff": float(total_n_eff),
        "star_graphs": aggregated,
    }


def aggregate_all(n_clients=5, phase3b_dir="data/phase3b",
                  output_dir="data/phase4"):
    """Main aggregation function."""
    os.makedirs(output_dir, exist_ok=True)

    log(f"{'='*60}")
    log("Phase 4.1 — Server cumulant aggregation")
    log(f"{'='*60}")

    clients = load_all_clients(n_clients, phase3b_dir)
    if not clients:
        log("  [ERROR] No clients loaded")
        return None

    aggregated = aggregate_cumulants(clients)

    # Save
    out_path = f"{output_dir}/global_cumulants.json"
    with open(out_path, "w") as f:
        json.dump(aggregated, f, indent=2)

    log(f"\n  Aggregated {len(aggregated['star_graphs'])} star graphs")
    for sg in aggregated["star_graphs"]:
        log(f"    Center={sg['center']}: {len(sg['cumulants'])} cumulants, "
            f"{len(sg['n_obs_pairs'])} pairs")

    log(f"\n  Saved to {out_path}")
    log(f"{'='*60}")

    return aggregated


if __name__ == "__main__":
    aggregate_all(n_clients=5)

Phase 4.1 — Server cumulant aggregation
Loaded 5 clients
Aggregating 5 clients, total n_eff = 4980.1

  Aggregated 11 star graphs
    Center=Raf: 6 cumulants, 1 pairs
    Center=Mek: 21 cumulants, 4 pairs
    Center=Plcg: 16 cumulants, 3 pairs
    Center=PIP2: 21 cumulants, 4 pairs
    Center=PIP3: 26 cumulants, 5 pairs
    Center=Erk: 16 cumulants, 3 pairs
    Center=Akt: 11 cumulants, 2 pairs
    Center=PKA: 26 cumulants, 5 pairs
    Center=PKC: 21 cumulants, 4 pairs
    Center=Jnk: 21 cumulants, 4 pairs
    Center=p38: 16 cumulants, 3 pairs

  Saved to data/phase4/global_cumulants.json


# tune_sachs.py

In [47]:
"""
Tune Phase 4.2 parameters on Sachs data.
Tries multiple (tau_ratio, c3_threshold) combinations.
Reports SHD and F1 for each.
"""

import json
import os
import numpy as np
import networkx as nx
import itertools


def log(msg):
    print(msg, flush=True)


# ============================================================
# GROUND TRUTH
# ============================================================

SACHS_GROUND_TRUTH_EDGES = [
    ("Raf", "Mek"), ("Mek", "Plcg"), ("Mek", "PIP2"), ("Mek", "PIP3"),
    ("Plcg", "PIP2"), ("Plcg", "PIP3"), ("PIP3", "PIP2"), ("PIP3", "Akt"),
    ("PIP2", "PKC"), ("PIP3", "PKC"), ("Akt", "PKA"), ("PKC", "PKA"),
    ("PKC", "Jnk"), ("PKA", "Jnk"), ("PKA", "p38"), ("PKA", "Erk"),
    ("Jnk", "p38"), ("Jnk", "Erk"), ("p38", "Erk"),
]

SACHS_VARIABLES = [
    "Raf", "Mek", "Plcg", "PIP2", "PIP3",
    "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"
]


# ============================================================
# CORE FUNCTIONS (reuse from Phase 4.2)
# ============================================================

def compute_tau(cumulants, x, y):
    c3_x = cumulants.get(f"C3_{x}", 0.0)
    c3_y = cumulants.get(f"C3_{y}", 0.0)
    c12_xy = cumulants.get(f"C12_{x}_{y}", 0.0)
    c21_xy = cumulants.get(f"C21_{x}_{y}", 0.0)
    c12_yx = cumulants.get(f"C12_{y}_{x}", 0.0)
    c21_yx = cumulants.get(f"C21_{y}_{x}", 0.0)
    tau_xy = abs(c3_x * c12_xy - c21_xy * c12_yx)
    tau_yx = abs(c3_y * c12_yx - c21_yx * c12_xy)
    return tau_xy, tau_yx


def find_source(cumulants, variables, tau_threshold_ratio=0.2):
    n = len(variables)
    tau_matrix = {}
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            x, y = variables[i], variables[j]
            tau_xy, _ = compute_tau(cumulants, x, y)
            tau_matrix[(x, y)] = tau_xy
    if not tau_matrix:
        return None, tau_matrix
    max_tau = max(tau_matrix.values())
    threshold = tau_threshold_ratio * max_tau
    for s in variables:
        all_small = True
        for j in variables:
            if j == s:
                continue
            if tau_matrix.get((s, j), float('inf')) > threshold:
                all_small = False
                break
        if all_small:
            return s, tau_matrix
    return None, tau_matrix


def eliminate_source(cumulants, source, descendants):
    updated = dict(cumulants)
    c3_s = cumulants.get(f"C3_{source}", 0.0)
    if abs(c3_s) < 1e-10:
        return updated
    a_is = {}
    for i in descendants:
        if i == source:
            continue
        c21_is = cumulants.get(f"C21_{i}_{source}", 0.0)
        a_is[i] = c21_is / c3_s
    for i in descendants:
        if i == source:
            continue
        c3_i = cumulants.get(f"C3_{i}", 0.0)
        if i in a_is:
            updated[f"C3_{i}"] = c3_i - (a_is[i] ** 3) * c3_s
    for i in descendants:
        if i == source:
            continue
        for j in descendants:
            if j == source or i == j:
                continue
            c12_ij = cumulants.get(f"C12_{i}_{j}", 0.0)
            c21_ij = cumulants.get(f"C21_{i}_{j}", 0.0)
            if i in a_is and j in a_is:
                updated[f"C12_{i}_{j}"] = c12_ij - a_is[i] * (a_is[j] ** 2) * c3_s
                updated[f"C21_{i}_{j}"] = c21_ij - (a_is[i] ** 2) * a_is[j] * c3_s
    return updated


def recursive_source_recovery(cumulants, variables, tau_threshold_ratio=0.2):
    causal_order = []
    remaining = list(variables)
    current = dict(cumulants)
    while len(remaining) > 1:
        source, _ = find_source(current, remaining, tau_threshold_ratio)
        if source is None:
            break
        causal_order.append(source)
        remaining.remove(source)
        if len(remaining) > 0:
            current = eliminate_source(current, source, remaining)
    if len(remaining) == 1:
        causal_order.append(remaining[0])
    return causal_order


def is_identifiable(star_graph, c3_threshold=0.05):
    variables = [star_graph["center"]] + star_graph["peripherals"]
    cumulants = star_graph["cumulants"]
    weak_vars = []
    for v in variables:
        c3 = cumulants.get(f"C3_{v}", 0.0)
        if abs(c3) < c3_threshold:
            weak_vars.append((v, c3))
    return len(weak_vars) == 0, weak_vars


# ============================================================
# BUILD GRAPH FROM RECURSIVE RESULTS
# ============================================================

def run_recursive_with_params(global_cum, tau_ratio, c3_threshold):
    """Run recursive source recovery with given params."""
    results = []
    skipped = []
    for sg in global_cum["star_graphs"]:
        center = sg["center"]
        peripherals = sg["peripherals"]
        variables = [center] + peripherals
        identifiable, weak_vars = is_identifiable(sg, c3_threshold)
        if not identifiable:
            skipped.append({"center": center, "weak_vars": weak_vars})
            continue
        causal_order = recursive_source_recovery(
            sg["cumulants"], variables, tau_ratio)
        results.append({
            "center": center,
            "peripherals": peripherals,
            "causal_order": causal_order,
        })
    return results, skipped


def extract_votes(results):
    """Extract direction votes from causal orders."""
    votes = {}
    for sg in results:
        center = sg["center"]
        peripherals = sg["peripherals"]
        order = sg["causal_order"]
        weight = len(peripherals)
        for i in range(len(order)):
            for j in range(i + 1, len(order)):
                u, v = order[i], order[j]
                votes.setdefault((u, v), []).append(weight)
    return votes


def aggregate_votes(votes):
    """Aggregate votes into edge decisions."""
    edges = set()
    for (u, v) in votes.keys():
        edges.add(tuple(sorted([u, v])))
    decisions = {}
    for (a, b) in edges:
        score_ab = sum(votes.get((a, b), []))
        score_ba = sum(votes.get((b, a), []))
        if score_ab > score_ba:
            decisions[(a, b)] = (a, b)
        elif score_ba > score_ab:
            decisions[(a, b)] = (b, a)
        else:
            decisions[(a, b)] = None
    return decisions


def build_graph(decisions):
    G = nx.DiGraph()
    G.add_nodes_from(SACHS_VARIABLES)
    for (a, b), direction in decisions.items():
        if direction is not None:
            G.add_edge(*direction)
    return G


def compute_metrics(G):
    true_G = nx.DiGraph()
    true_G.add_nodes_from(SACHS_VARIABLES)
    true_G.add_edges_from(SACHS_GROUND_TRUTH_EDGES)

    est_edges = set(G.edges())
    true_edges = set(true_G.edges())

    tp = len(est_edges & true_edges)
    fp = len(est_edges - true_edges)
    fn = len(true_edges - est_edges)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    # SHD
    missing = true_edges - est_edges
    extra = est_edges - true_edges
    reversed_cnt = 0
    for (u, v) in extra:
        if (v, u) in missing:
            reversed_cnt += 1
    shd = len(missing) + len(extra)

    # Skeleton F1
    est_skel = set(tuple(sorted(e)) for e in est_edges)
    true_skel = set(tuple(sorted(e)) for e in true_edges)
    tp_s = len(est_skel & true_skel)
    fp_s = len(est_skel - true_skel)
    fn_s = len(true_skel - est_skel)
    p_s = tp_s / (tp_s + fp_s) if (tp_s + fp_s) > 0 else 0.0
    r_s = tp_s / (tp_s + fn_s) if (tp_s + fn_s) > 0 else 0.0
    f1_s = 2 * p_s * r_s / (p_s + r_s) if (p_s + r_s) > 0 else 0.0

    return {
        "shd": shd, "missing": len(missing), "extra": len(extra),
        "reversed": reversed_cnt,
        "f1": f1, "precision": precision, "recall": recall,
        "tp": tp, "fp": fp, "fn": fn,
        "f1_skeleton": f1_s, "p_skeleton": p_s, "r_skeleton": r_s,
        "n_edges": len(est_edges),
    }


# ============================================================
# TUNING LOOP
# ============================================================

def tune():
    with open("data/phase4/global_cumulants.json") as f:
        global_cum = json.load(f)

    # Grid of parameters
    tau_ratios = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
    c3_thresholds = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.08, 0.1]

    results = []

    log(f"{'tau_ratio':>10} {'c3_thr':>8} {'n_ident':>8} {'n_edges':>8} "
        f"{'SHD':>6} {'F1':>6} {'F1_skel':>8} {'P':>6} {'R':>6}")
    log("=" * 80)

    for tau_ratio, c3_thr in itertools.product(tau_ratios, c3_thresholds):
        rec_results, skipped = run_recursive_with_params(
            global_cum, tau_ratio, c3_thr)
        votes = extract_votes(rec_results)
        decisions = aggregate_votes(votes)
        G = build_graph(decisions)
        m = compute_metrics(G)

        # ← results.append() MUST be INSIDE the loop
        results.append({
            "tau_ratio": tau_ratio,
            "c3_threshold": c3_thr,
            "n_identifiable": len(rec_results),
            "n_skipped": len(skipped),
            "causal_orders": [r["causal_order"] for r in rec_results],
            **m,
        })

        log(f"{tau_ratio:>10.2f} {c3_thr:>8.2f} {len(rec_results):>8} "
            f"{m['n_edges']:>8} {m['shd']:>6} {m['f1']:>6.3f} "
            f"{m['f1_skeleton']:>8.3f} {m['precision']:>6.3f} {m['recall']:>6.3f}")
        

    # Find best configurations
    log("\n" + "=" * 80)
    log("TOP 5 by F1 (direction-aware):")
    log("=" * 80)
    sorted_by_f1 = sorted(results, key=lambda x: -x["f1"])[:5]
    for r in sorted_by_f1:
        log(f"  tau_ratio={r['tau_ratio']:.2f}, c3_thr={r['c3_threshold']:.2f}: "
            f"F1={r['f1']:.3f}, SHD={r['shd']}, "
            f"P={r['precision']:.3f}, R={r['recall']:.3f}")

    log("\n" + "=" * 80)
    log("TOP 5 by Skeleton F1:")
    log("=" * 80)
    sorted_by_skel = sorted(results, key=lambda x: -x["f1_skeleton"])[:5]
    for r in sorted_by_skel:
        log(f"  tau_ratio={r['tau_ratio']:.2f}, c3_thr={r['c3_threshold']:.2f}: "
            f"F1_skel={r['f1_skeleton']:.3f}, SHD={r['shd']}, "
            f"P={r['p_skeleton']:.3f}, R={r['r_skeleton']:.3f}")

    log("\n" + "=" * 80)
    log("TOP 5 by lowest SHD:")
    log("=" * 80)
    sorted_by_shd = sorted(results, key=lambda x: x["shd"])[:5]
    for r in sorted_by_shd:
        log(f"  tau_ratio={r['tau_ratio']:.2f}, c3_thr={r['c3_threshold']:.2f}: "
            f"SHD={r['shd']}, F1={r['f1']:.3f}, F1_skel={r['f1_skeleton']:.3f}")

    # Save
    os.makedirs("data/phase5", exist_ok=True)
    with open("data/phase5/tuning_results.json", "w") as f:
        json.dump(results, f, indent=2)

    log(f"\nSaved {len(results)} results to data/phase5/tuning_results.json")
        
    return results


if __name__ == "__main__":
    tune()

 tau_ratio   c3_thr  n_ident  n_edges    SHD     F1  F1_skel      P      R
      0.05     0.00       11       10     19  0.345    0.552  0.500  0.263
      0.05     0.01        5        0     19  0.000    0.000  0.000  0.000
      0.05     0.02        5        0     19  0.000    0.000  0.000  0.000
      0.05     0.03        5        0     19  0.000    0.000  0.000  0.000
      0.05     0.04        5        0     19  0.000    0.000  0.000  0.000
      0.05     0.05        5        0     19  0.000    0.000  0.000  0.000
      0.05     0.08        1        0     19  0.000    0.000  0.000  0.000
      0.05     0.10        1        0     19  0.000    0.000  0.000  0.000
      0.10     0.00       11       11     18  0.400    0.600  0.545  0.316
      0.10     0.01        5        1     18  0.100    0.100  1.000  0.053
      0.10     0.02        5        1     18  0.100    0.100  1.000  0.053
      0.10     0.03        5        1     18  0.100    0.100  1.000  0.053
      0.10     0.04      

# save_best_graph.py

In [49]:
"""
Phase 6.1 — Save the best graph from tuning results.
"""

import json
import networkx as nx
import os


def log(msg):
    print(msg, flush=True)


SACHS_GROUND_TRUTH_EDGES = [
    ("Raf", "Mek"), ("Mek", "Plcg"), ("Mek", "PIP2"), ("Mek", "PIP3"),
    ("Plcg", "PIP2"), ("Plcg", "PIP3"), ("PIP3", "PIP2"), ("PIP3", "Akt"),
    ("PIP2", "PKC"), ("PIP3", "PKC"), ("Akt", "PKA"), ("PKC", "PKA"),
    ("PKC", "Jnk"), ("PKA", "Jnk"), ("PKA", "p38"), ("PKA", "Erk"),
    ("Jnk", "p38"), ("Jnk", "Erk"), ("p38", "Erk"),
]

SACHS_VARIABLES = [
    "Raf", "Mek", "Plcg", "PIP2", "PIP3",
    "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"
]


def save_best():
    os.makedirs("data/phase6", exist_ok=True)

    with open("data/phase5/tuning_results.json") as f:
        results = json.load(f)

    # Best by SHD
    best = min(results, key=lambda x: x["shd"])

    log(f"{'='*60}")
    log("BEST CONFIG (by SHD)")
    log(f"{'='*60}")
    log(f"  tau_ratio = {best['tau_ratio']}")
    log(f"  c3_threshold = {best['c3_threshold']}")
    log(f"  SHD = {best['shd']}")
    log(f"  F1 = {best['f1']:.3f}")
    log(f"  F1_skeleton = {best['f1_skeleton']:.3f}")
    log(f"  Precision = {best['precision']:.3f}")
    log(f"  Recall = {best['recall']:.3f}")
    log(f"  n_edges = {best['n_edges']}")

    # Rebuild graph from causal orders
    G = nx.DiGraph()
    G.add_nodes_from(SACHS_VARIABLES)

    for co in best["causal_orders"]:
        # Each causal_order is a list; build edges from consecutive pairs
        for i in range(len(co) - 1):
            G.add_edge(co[i], co[i + 1])

    log(f"\n  Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    log(f"\n  Estimated edges:")
    for u, v in sorted(G.edges()):
        log(f"    {u} -> {v}")

    # Ground truth
    true_G = nx.DiGraph()
    true_G.add_nodes_from(SACHS_VARIABLES)
    true_G.add_edges_from(SACHS_GROUND_TRUTH_EDGES)

    # Compare
    est_edges = set(G.edges())
    true_edges = set(true_G.edges())
    tp = est_edges & true_edges
    fp = est_edges - true_edges
    fn = true_edges - est_edges

    log(f"\n  Correct (TP): {len(tp)}")
    for e in sorted(tp):
        log(f"    ✓ {e[0]} -> {e[1]}")

    log(f"\n  Extra (FP): {len(fp)}")
    for e in sorted(fp):
        log(f"    ✗ {e[0]} -> {e[1]}")

    log(f"\n  Missing (FN): {len(fn)}")
    for e in sorted(fn):
        log(f"    ✗ {e[0]} -> {e[1]}")

    # Save
    with open("data/phase6/best_graph.json", "w") as f:
        json.dump({
            "config": {
                "tau_ratio": best["tau_ratio"],
                "c3_threshold": best["c3_threshold"],
            },
            "metrics": {
                "shd": best["shd"],
                "f1": best["f1"],
                "f1_skeleton": best["f1_skeleton"],
                "precision": best["precision"],
                "recall": best["recall"],
                "n_edges": best["n_edges"],
            },
            "estimated_edges": sorted([list(e) for e in G.edges()]),
            "true_edges": sorted([list(e) for e in true_G.edges()]),
            "correct": sorted([list(e) for e in tp]),
            "extra": sorted([list(e) for e in fp]),
            "missing": sorted([list(e) for e in fn]),
            "causal_orders": best["causal_orders"],
        }, f, indent=2)

    log(f"\n  Saved to data/phase6/best_graph.json")
    log(f"{'='*60}")


if __name__ == "__main__":
    save_best()

BEST CONFIG (by SHD)
  tau_ratio = 0.3
  c3_threshold = 0.0
  SHD = 15
  F1 = 0.615
  F1_skeleton = 0.769
  Precision = 0.600
  Recall = 0.632
  n_edges = 20

  Graph: 11 nodes, 17 edges

  Estimated edges:
    Akt -> PKC
    Erk -> Jnk
    Erk -> PKA
    Jnk -> Erk
    Jnk -> p38
    Mek -> Akt
    Mek -> PIP2
    PIP2 -> PIP3
    PIP2 -> Plcg
    PIP3 -> Mek
    PIP3 -> PKC
    PKA -> Jnk
    PKA -> PKC
    PKA -> p38
    PKC -> p38
    Plcg -> PIP2
    p38 -> Erk

  Correct (TP): 8
    ✓ Jnk -> Erk
    ✓ Jnk -> p38
    ✓ Mek -> PIP2
    ✓ PIP3 -> PKC
    ✓ PKA -> Jnk
    ✓ PKA -> p38
    ✓ Plcg -> PIP2
    ✓ p38 -> Erk

  Extra (FP): 9
    ✗ Akt -> PKC
    ✗ Erk -> Jnk
    ✗ Erk -> PKA
    ✗ Mek -> Akt
    ✗ PIP2 -> PIP3
    ✗ PIP2 -> Plcg
    ✗ PIP3 -> Mek
    ✗ PKA -> PKC
    ✗ PKC -> p38

  Missing (FN): 11
    ✗ Akt -> PKA
    ✗ Mek -> PIP3
    ✗ Mek -> Plcg
    ✗ PIP2 -> PKC
    ✗ PIP3 -> Akt
    ✗ PIP3 -> PIP2
    ✗ PKA -> Erk
    ✗ PKC -> Jnk
    ✗ PKC -> PKA
    ✗ Plcg -> PI

# File — run_baselines.py


In [50]:
"""
Phase 6.2 — Run baselines on Sachs data.
Baselines: DirectLiNGAM + deletion, DirectLiNGAM + mean imputation.
"""

import json
import os
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.linear_model import LinearRegression


def log(msg):
    print(msg, flush=True)


SACHS_GROUND_TRUTH_EDGES = [
    ("Raf", "Mek"), ("Mek", "Plcg"), ("Mek", "PIP2"), ("Mek", "PIP3"),
    ("Plcg", "PIP2"), ("Plcg", "PIP3"), ("PIP3", "PIP2"), ("PIP3", "Akt"),
    ("PIP2", "PKC"), ("PIP3", "PKC"), ("Akt", "PKA"), ("PKC", "PKA"),
    ("PKC", "Jnk"), ("PKA", "Jnk"), ("PKA", "p38"), ("PKA", "Erk"),
    ("Jnk", "p38"), ("Jnk", "Erk"), ("p38", "Erk"),
]

SACHS_VARIABLES = [
    "Raf", "Mek", "Plcg", "PIP2", "PIP3",
    "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"
]


def direct_lingam_simple(df, variables):
    """
    Simplified DirectLiNGAM using pairwise regression + residual independence.
    """
    n = len(variables)
    direction_scores = {}

    for i in range(n):
        for j in range(i + 1, n):
            x = df[variables[i]].values
            y = df[variables[j]].values

            # Regress y on x
            lr_yx = LinearRegression().fit(x.reshape(-1, 1), y)
            res_yx = y - lr_yx.predict(x.reshape(-1, 1))

            # Regress x on y
            lr_xy = LinearRegression().fit(y.reshape(-1, 1), x)
            res_xy = x - lr_xy.predict(y.reshape(-1, 1))

            # Independence via correlation with parent
            corr_yx = abs(np.corrcoef(x, res_yx)[0, 1])
            corr_xy = abs(np.corrcoef(y, res_xy)[0, 1])

            if corr_yx < corr_xy:
                direction_scores[(variables[i], variables[j])] = corr_yx
                direction_scores[(variables[j], variables[i])] = corr_xy
            else:
                direction_scores[(variables[i], variables[j])] = corr_xy
                direction_scores[(variables[j], variables[i])] = corr_yx

    G = nx.DiGraph()
    G.add_nodes_from(variables)

    for i in range(n):
        for j in range(i + 1, n):
            u, v = variables[i], variables[j]
            score_uv = direction_scores.get((u, v), 1.0)
            score_vu = direction_scores.get((v, u), 1.0)
            if score_uv < score_vu:
                G.add_edge(u, v)
            elif score_vu < score_uv:
                G.add_edge(v, u)

    return G


def compute_metrics(G):
    """Compute SHD + F1 against ground truth."""
    true_G = nx.DiGraph()
    true_G.add_nodes_from(SACHS_VARIABLES)
    true_G.add_edges_from(SACHS_GROUND_TRUTH_EDGES)

    est_edges = set(G.edges())
    true_edges = set(true_G.edges())

    tp = len(est_edges & true_edges)
    fp = len(est_edges - true_edges)
    fn = len(true_edges - est_edges)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    missing = true_edges - est_edges
    extra = est_edges - true_edges
    shd = len(missing) + len(extra)

    return {
        "shd": shd, "f1": f1, "precision": precision, "recall": recall,
        "n_edges": len(est_edges), "tp": tp, "fp": fp, "fn": fn,
    }


def run_baseline(name, df, variables):
    """Run a baseline and return metrics."""
    log(f"\n{'='*60}")
    log(f"Baseline: {name}")
    log(f"{'='*60}")

    G = direct_lingam_simple(df, variables)
    metrics = compute_metrics(G)

    log(f"  SHD = {metrics['shd']}")
    log(f"  F1 = {metrics['f1']:.3f}")
    log(f"  Precision = {metrics['precision']:.3f}")
    log(f"  Recall = {metrics['recall']:.3f}")
    log(f"  n_edges = {metrics['n_edges']}")

    return {"name": name, **metrics}


def main():
    os.makedirs("data/phase6", exist_ok=True)

    # Load all clients and concatenate
    log("Loading all clients...")
    dfs = []
    for k in range(5):
        df = pd.read_csv(f"data/clients/client_{k}.csv")
        dfs.append(df)

    all_data = pd.concat(dfs, ignore_index=True)
    log(f"Total rows: {len(all_data)}")

    results = []

    # Baseline 1: DirectLiNGAM + deletion (complete rows only)
    complete_data = all_data[SACHS_VARIABLES].dropna()
    log(f"Complete rows: {len(complete_data)} / {len(all_data)}")

    if len(complete_data) > 50:
        results.append(run_baseline(
            "DirectLiNGAM + deletion", complete_data, SACHS_VARIABLES))
    else:
        log("[Skip] Not enough complete rows for deletion baseline")

    # Baseline 2: Mean imputation
    imputed_data = all_data[SACHS_VARIABLES].copy()
    for col in imputed_data.columns:
        imputed_data[col] = imputed_data[col].fillna(imputed_data[col].mean())
    results.append(run_baseline(
        "DirectLiNGAM + mean imputation", imputed_data, SACHS_VARIABLES))

    # Our method (from best config)
    with open("data/phase5/tuning_results.json") as f:
        tuning = json.load(f)
    best = min(tuning, key=lambda x: x["shd"])

    G_ours = nx.DiGraph()
    G_ours.add_nodes_from(SACHS_VARIABLES)
    for co in best["causal_orders"]:
        for i in range(len(co) - 1):
            G_ours.add_edge(co[i], co[i + 1])

    metrics_ours = compute_metrics(G_ours)
    log(f"\n{'='*60}")
    log(f"Our method (tuned, τ={best['tau_ratio']}, c3={best['c3_threshold']})")
    log(f"{'='*60}")
    log(f"  SHD = {metrics_ours['shd']}")
    log(f"  F1 = {metrics_ours['f1']:.3f}")
    log(f"  Precision = {metrics_ours['precision']:.3f}")
    log(f"  Recall = {metrics_ours['recall']:.3f}")
    log(f"  n_edges = {metrics_ours['n_edges']}")

    results.append({"name": "Our method (tuned)", **metrics_ours})

    # Comparison table
    log(f"\n{'='*60}")
    log("COMPARISON TABLE")
    log(f"{'='*60}")
    log(f"{'Method':<35} {'SHD':>5} {'F1':>6} {'P':>6} {'R':>6} {'Edges':>6}")
    log("-" * 70)
    for r in results:
        log(f"{r['name']:<35} {r['shd']:>5} {r['f1']:>6.3f} "
            f"{r['precision']:>6.3f} {r['recall']:>6.3f} {r['n_edges']:>6}")

    # Save
    with open("data/phase6/baseline_comparison.json", "w") as f:
        json.dump(results, f, indent=2)

    log(f"\nSaved to data/phase6/baseline_comparison.json")


if __name__ == "__main__":
    main()

Loading all clients...
Total rows: 5400
Complete rows: 2394 / 5400

Baseline: DirectLiNGAM + deletion
  SHD = 46
  F1 = 0.378
  Precision = 0.255
  Recall = 0.737
  n_edges = 55

Baseline: DirectLiNGAM + mean imputation
  SHD = 46
  F1 = 0.378
  Precision = 0.255
  Recall = 0.737
  n_edges = 55

Our method (tuned, τ=0.3, c3=0.0)
  SHD = 20
  F1 = 0.444
  Precision = 0.471
  Recall = 0.421
  n_edges = 17

COMPARISON TABLE
Method                                SHD     F1      P      R  Edges
----------------------------------------------------------------------
DirectLiNGAM + deletion                46  0.378  0.255  0.737     55
DirectLiNGAM + mean imputation         46  0.378  0.255  0.737     55
Our method (tuned)                     20  0.444  0.471  0.421     17

Saved to data/phase6/baseline_comparison.json


# run_all_K.py

>This applies to all client counts from 1 to 6 altogether. The earlier setting was only for client = 5. Do not forget to download the Sachs dataset using the first code block mentioned above, and then the following can be run.

In [9]:
"""
fed_cd_pipeline.py — Federated Causal Discovery Pipeline (Cumulant-Based)
Runs for K = 1..6 clients, saves to separate directories.
F1-based tuning (fixes SHD-only bias).
Notebook-friendly — no argparse, no __main__ block.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import itertools


def log(msg):
    print(msg, flush=True)


# ============================================================
# CONSTANTS
# ============================================================

VARS = [
    "Raf", "Mek", "Plcg", "PIP2", "PIP3",
    "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"
]

TRUE_EDGES = [
    ("Raf", "Mek"), ("Mek", "Plcg"), ("Mek", "PIP2"), ("Mek", "PIP3"),
    ("Plcg", "PIP2"), ("Plcg", "PIP3"), ("PIP3", "PIP2"), ("PIP3", "Akt"),
    ("PIP2", "PKC"), ("PIP3", "PKC"), ("Akt", "PKA"), ("PKC", "PKA"),
    ("PKC", "Jnk"), ("PKA", "Jnk"), ("PKA", "p38"), ("PKA", "Erk"),
    ("Jnk", "p38"), ("Jnk", "Erk"), ("p38", "Erk"),
]

TAU_RATIOS = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
C3_THRESHOLDS = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.08, 0.1]
BASELINE_PRUNE = 0.05


# ============================================================
# PHASE 1 — SPLIT + INJECT MISSINGNESS
# ============================================================

def _inject_mcar(df, rate, seed):
    rng = np.random.RandomState(seed)
    mask = rng.rand(*df.shape) < rate
    out = df.copy()
    out[mask] = np.nan
    return out


def _inject_mar(df, target, parents, rate, seed):
    rng = np.random.RandomState(seed)
    out = df.copy()
    X = df[parents].values.astype(float)
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
    beta = rng.randn(len(parents)) * 0.5
    logits = X @ beta
    logits = logits - np.percentile(logits, 100 * (1 - rate))
    probs = 1 / (1 + np.exp(-logits))
    mask = rng.rand(len(df)) < probs
    out.loc[mask, target] = np.nan
    return out


def phase1_split(K, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv("data/sachs.csv")
    df.columns = df.columns.str.strip()
    df = df.rename(columns={c: "p38" for c in df.columns if c.lower() == "p38"})
    df = df[VARS].copy()

    rng = np.random.RandomState(42)
    idx = rng.permutation(len(df))
    splits = np.array_split(idx, K)

    log(f"[K={K}] Split {len(df)} rows into {K} clients")

    for k, split in enumerate(splits):
        cdf = df.iloc[split].reset_index(drop=True)
        if K == 1:
            cdf = _inject_mar(cdf, "Erk", ["PKA", "Jnk", "p38"], 0.30, seed=42)
        elif k % 3 == 0:
            cdf = _inject_mcar(cdf, 0.05, seed=42 + k)
        elif k % 3 == 1:
            cdf = _inject_mar(cdf, "Erk", ["PKA", "Jnk", "p38"], 0.30, seed=42 + k)
        else:
            cdf = _inject_mar(cdf, "Akt", ["PIP3"], 0.25, seed=42 + k)

        cdf.to_csv(f"{output_dir}/client_{k}.csv", index=False)
        log(f"  Client {k}: {len(cdf)} rows, {cdf.isna().mean().mean():.1%} missing")


# ============================================================
# PHASE 2 — CLASSIFY + FIT MISSINGNESS MODELS
# ============================================================

def _classify(df, cid):
    rates = df.isna().mean()
    V_o = [c for c in df.columns if rates[c] < 0.05]
    V_m = [c for c in df.columns if rates[c] >= 0.05]
    return {
        "client": cid, "V_o": V_o, "V_m": V_m,
        "missing_rates": {c: float(rates[c]) for c in df.columns},
        "n_rows": len(df),
    }


def _fit_model(df, target, parents, cid):
    R = df[target].isna().astype(int).values
    rate = R.mean()
    if rate < 0.05 or R.sum() < 10 or R.sum() == len(R):
        return None
    if len(parents) == 0:
        parents = [c for c in df.columns if c != target]
    pdata = df[parents]
    mask = pdata.notna().all(axis=1).values
    if mask.sum() < 50:
        return None
    X = pdata.values[mask].astype(float)
    R_obs = R[mask]
    if R_obs.sum() < 10 or R_obs.sum() == len(R_obs):
        return None
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    model = LogisticRegression(penalty="l2", C=1.0, max_iter=1000,
                                random_state=42, solver="lbfgs")
    model.fit(Xs, R_obs)
    probs = model.predict_proba(Xs)[:, 1]
    auc = roc_auc_score(R_obs, probs)
    return {
        "client": cid, "target": target, "parents": parents,
        "missing_rate": float(rate), "auc": float(auc),
        "model": model, "scaler": scaler,
    }


def phase2_classify(K, client_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for k in range(K):
        df = pd.read_csv(f"{client_dir}/client_{k}.csv")
        cls = _classify(df, k)
        with open(f"{output_dir}/classification_{k}.json", "w") as f:
            json.dump(cls, f, indent=2)
        for target in cls["V_m"]:
            m = _fit_model(df, target, cls["V_o"], k)
            if m is None:
                continue
            with open(f"{output_dir}/model_{k}_{target}.pkl", "wb") as f:
                pickle.dump({"model": m["model"], "scaler": m["scaler"]}, f)
            meta = {kk: vv for kk, vv in m.items() if kk not in ["model", "scaler"]}
            with open(f"{output_dir}/meta_{k}_{target}.json", "w") as f:
                json.dump(meta, f, indent=2)


# ============================================================
# PHASE 3 — IPW CORRECTION
# ============================================================

def _weights_col(df, target, model_dict):
    model, scaler, parents = model_dict["model"], model_dict["scaler"], model_dict["parents"]
    obs = df[target].notna().values
    w = np.ones(len(df))
    if obs.sum() == 0:
        return w
    pdata = df.loc[obs, parents]
    if pdata.isna().any().any():
        pdata = pdata.fillna(pdata.mean())
    Xs = scaler.transform(pdata.values.astype(float))
    pm = np.clip(model.predict_proba(Xs)[:, 1], 1e-6, 1 - 1e-6)
    ww = np.minimum(1.0 / (1 - pm), np.percentile(1.0 / (1 - pm), 95))
    w[obs] = ww
    return w


def phase3_ipw(K, client_dir, phase2_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for k in range(K):
        df = pd.read_csv(f"{client_dir}/client_{k}.csv")
        with open(f"{phase2_dir}/classification_{k}.json") as f:
            cls = json.load(f)
        w = np.ones(len(df))
        for target in cls["V_m"]:
            path = f"{phase2_dir}/model_{k}_{target}.pkl"
            if not os.path.exists(path):
                continue
            with open(path, "rb") as f:
                md = pickle.load(f)
            with open(f"{phase2_dir}/meta_{k}_{target}.json") as f:
                md.update(json.load(f))
            w *= _weights_col(df, target, md)
        w = w / w.mean()
        n_eff = (w.sum() ** 2) / (w ** 2).sum()
        pd.DataFrame({"row": np.arange(len(df)), "weight": w}).to_csv(
            f"{output_dir}/weights_{k}.csv", index=False)
        with open(f"{output_dir}/ipw_meta_{k}.json", "w") as f:
            json.dump({"client": k, "n_rows": len(df), "n_eff": float(n_eff)}, f, indent=2)
        log(f"  [K={K}] Client {k}: n_eff={n_eff:.1f}/{len(df)}")


# ============================================================
# PHASE 3B — STAR GRAPHS + WEIGHTED CUMULANTS
# ============================================================

def _skeleton():
    G = nx.Graph()
    G.add_nodes_from(VARS)
    for u, v in TRUE_EDGES:
        G.add_edge(u, v)
    return G


def _star_graphs(skel):
    out = []
    for c in VARS:
        if c not in skel: continue
        periph = list(skel.neighbors(c))
        if periph:
            out.append({"center": c, "peripherals": periph})
    return out


def _c3(x, w):
    mu = np.sum(w * x) / np.sum(w)
    return np.sum(w * (x - mu) ** 3) / np.sum(w)


def _c12(x, y, w):
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) * (y - my) ** 2) / np.sum(w)


def _c21(x, y, w):
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) ** 2 * (y - my)) / np.sum(w)


def _pair_cums(df, w, x, y, min_obs=50):
    mask = df[x].notna() & df[y].notna()
    n = mask.sum()
    if n < min_obs: return None
    xv = df.loc[mask, x].values.astype(float)
    yv = df.loc[mask, y].values.astype(float)
    ww = w[mask] * (n / w[mask].sum())
    return {
        "C3_x": float(_c3(xv, ww)), "C3_y": float(_c3(yv, ww)),
        "C12_xy": float(_c12(xv, yv, ww)), "C21_xy": float(_c21(xv, yv, ww)),
        "C12_yx": float(_c12(yv, xv, ww)), "C21_yx": float(_c21(yv, xv, ww)),
        "n_obs": int(n),
    }


def phase3b_cumulants(K, client_dir, phase3_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    sgs = _star_graphs(_skeleton())
    for k in range(K):
        df = pd.read_csv(f"{client_dir}/client_{k}.csv")
        w = pd.read_csv(f"{phase3_dir}/weights_{k}.csv")["weight"].values
        client_cum = {"client": k, "n_rows": len(df), "star_graphs": []}
        for sg in sgs:
            c, periph = sg["center"], sg["peripherals"]
            cums, nobs = {}, {}
            for p in periph:
                pc = _pair_cums(df, w, c, p)
                if pc is None: continue
                cums[f"C3_{c}"] = pc["C3_x"]
                cums[f"C3_{p}"] = pc["C3_y"]
                cums[f"C12_{c}_{p}"] = pc["C12_xy"]
                cums[f"C21_{c}_{p}"] = pc["C21_xy"]
                cums[f"C12_{p}_{c}"] = pc["C12_yx"]
                cums[f"C21_{p}_{c}"] = pc["C21_yx"]
                nobs[f"{c}_{p}"] = pc["n_obs"]
            client_cum["star_graphs"].append({
                "center": c, "peripherals": periph,
                "cumulants": cums, "n_obs_pairs": nobs,
            })
        with open(f"{output_dir}/cumulants_{k}.json", "w") as f:
            json.dump(client_cum, f, indent=2)


def phase3b_package(K, phase3_dir, phase3b_dir, output_dir):
    for k in range(K):
        with open(f"{phase3b_dir}/cumulants_{k}.json") as f:
            cum = json.load(f)
        with open(f"{phase3_dir}/ipw_meta_{k}.json") as f:
            ipw = json.load(f)
        packaged = {
            "client": k, "n_rows": cum["n_rows"], "n_eff": ipw["n_eff"],
            "n_eff_ratio": ipw["n_eff"] / ipw["n_rows"],
            "star_graphs": cum["star_graphs"],
        }
        with open(f"{output_dir}/client_cumulants_{k}.json", "w") as f:
            json.dump(packaged, f, indent=2)


# ============================================================
# PHASE 4 — AGGREGATE
# ============================================================

def phase4_aggregate(K, phase3b_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    clients = []
    for k in range(K):
        with open(f"{phase3b_dir}/client_cumulants_{k}.json") as f:
            clients.append(json.load(f))
    total_neff = sum(c["n_eff"] for c in clients)
    by_center = {}
    for c in clients:
        for sg in c["star_graphs"]:
            by_center.setdefault(sg["center"], []).append((c, sg))
    agg = []
    for center, sg_list in by_center.items():
        periph = sg_list[0][1]["peripherals"]
        cums, nobs = {}, {}
        for c, sg in sg_list:
            wt = c["n_eff"] / total_neff
            for key, val in sg["cumulants"].items():
                cums.setdefault(key, 0.0)
                cums[key] += wt * val
            for pk, n in sg["n_obs_pairs"].items():
                nobs.setdefault(pk, 0)
                nobs[pk] += n
        agg.append({"center": center, "peripherals": periph,
                     "cumulants": cums, "n_obs_pairs": nobs})
    with open(f"{output_dir}/global_cumulants.json", "w") as f:
        json.dump({"n_clients": K, "total_n_eff": float(total_neff),
                    "star_graphs": agg}, f, indent=2)


# ============================================================
# PHASE 5 — TUNE (F1-BASED)
# ============================================================

def _tau(cums, x, y):
    c3x = cums.get(f"C3_{x}", 0.0); c3y = cums.get(f"C3_{y}", 0.0)
    c12xy = cums.get(f"C12_{x}_{y}", 0.0); c21xy = cums.get(f"C21_{x}_{y}", 0.0)
    c12yx = cums.get(f"C12_{y}_{x}", 0.0); c21yx = cums.get(f"C21_{y}_{x}", 0.0)
    return (abs(abs(c3x) * c12xy - c21xy * c12yx),
            abs(abs(c3y) * c12yx - c21yx * c12xy))


def _find_source(cums, vars_, ratio):
    n = len(vars_)
    tau = {}
    for i in range(n):
        for j in range(n):
            if i == j: continue
            t, _ = _tau(cums, vars_[i], vars_[j])
            tau[(vars_[i], vars_[j])] = t
    if not tau: return None
    thr = ratio * max(tau.values())
    for s in vars_:
        if all(tau.get((s, j), 1.0) <= thr for j in vars_ if j != s):
            return s
    return None


def _eliminate(cums, src, desc):
    upd = dict(cums)
    c3s = cums.get(f"C3_{src}", 0.0)
    if abs(c3s) < 1e-10: return upd
    a = {i: np.clip(cums.get(f"C21_{i}_{src}", 0.0) / c3s, -2, 2)
         for i in desc if i != src}
    for i in desc:
        if i == src: continue
        upd[f"C3_{i}"] = cums.get(f"C3_{i}", 0.0) - (a.get(i, 0) ** 3) * c3s
    for i in desc:
        if i == src: continue
        for j in desc:
            if j == src or i == j: continue
            ai, aj = a.get(i, 0), a.get(j, 0)
            upd[f"C12_{i}_{j}"] = cums.get(f"C12_{i}_{j}", 0.0) - ai * (aj ** 2) * c3s
            upd[f"C21_{i}_{j}"] = cums.get(f"C21_{i}_{j}", 0.0) - (ai ** 2) * aj * c3s
    return upd


def _recover(cums, vars_, ratio):
    order, rem, cur = [], list(vars_), dict(cums)
    while len(rem) > 1:
        s = _find_source(cur, rem, ratio)
        if s is None: break
        order.append(s); rem.remove(s)
        if rem: cur = _eliminate(cur, s, rem)
    if len(rem) == 1: order.append(rem[0])
    return order


def _identifiable(sg, thr):
    return all(abs(sg["cumulants"].get(f"C3_{v}", 0.0)) >= thr
               for v in [sg["center"]] + sg["peripherals"])


def _build_graph(orders):
    G = nx.DiGraph()
    G.add_nodes_from(VARS)
    for o in orders:
        for i in range(len(o) - 1):
            G.add_edge(o[i], o[i + 1])
    return G


def _metrics(G):
    true = nx.DiGraph()
    true.add_nodes_from(VARS)
    true.add_edges_from(TRUE_EDGES)
    est = set(G.edges()); tru = set(true.edges())
    tp = len(est & tru); fp = len(est - tru); fn = len(tru - est)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"shd": len(tru - est) + len(est - tru), "f1": f1,
            "precision": p, "recall": r, "n_edges": len(est),
            "tp": tp, "fp": fp, "fn": fn}


def phase5_tune(K, phase4_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{phase4_dir}/global_cumulants.json") as f:
        gc = json.load(f)
    results = []
    for ratio, thr in itertools.product(TAU_RATIOS, C3_THRESHOLDS):
        orders = []
        for sg in gc["star_graphs"]:
            if not _identifiable(sg, thr): continue
            o = _recover(sg["cumulants"], [sg["center"]] + sg["peripherals"], ratio)
            if o: orders.append(o)
        m = _metrics(_build_graph(orders))
        results.append({"tau_ratio": ratio, "c3_threshold": thr,
                         "causal_orders": orders, **m})
    # --- FIX: sort by F1 (highest first), not SHD ---
    results.sort(key=lambda x: -x["f1"])
    best = results[0]
    with open(f"{output_dir}/tuning_results.json", "w") as f:
        json.dump(results, f, indent=2)
    log(f"  [K={K}] Best (by F1): tau={best['tau_ratio']}, c3={best['c3_threshold']}, "
        f"SHD={best['shd']}, F1={best['f1']:.3f}")
    return best


# ============================================================
# PHASE 6 — SAVE BEST + BASELINES
# ============================================================

def _lingam_baseline(df):
    n = len(VARS)
    scores = {}
    for i in range(n):
        for j in range(i + 1, n):
            x, y = df[VARS[i]].values, df[VARS[j]].values
            r1 = y - LinearRegression().fit(x.reshape(-1, 1), y).predict(x.reshape(-1, 1))
            r2 = x - LinearRegression().fit(y.reshape(-1, 1), x).predict(y.reshape(-1, 1))
            scores[(VARS[i], VARS[j])] = abs(np.corrcoef(x, r1)[0, 1])
            scores[(VARS[j], VARS[i])] = abs(np.corrcoef(y, r2)[0, 1])
    G = nx.DiGraph()
    G.add_nodes_from(VARS)
    for i in range(n):
        for j in range(i + 1, n):
            u, v = VARS[i], VARS[j]
            suv, svu = scores.get((u, v), 1.0), scores.get((v, u), 1.0)
            if suv < BASELINE_PRUNE and suv < svu: G.add_edge(u, v)
            elif svu < BASELINE_PRUNE and svu < suv: G.add_edge(v, u)
    return G


def phase6_baselines(K, client_dir, phase5_dir, output_dir, best):
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/best_config.json", "w") as f:
        json.dump(best, f, indent=2)
    dfs = [pd.read_csv(f"{client_dir}/client_{k}.csv") for k in range(K)]
    all_data = pd.concat(dfs, ignore_index=True)
    results = []
    complete = all_data[VARS].dropna()
    if len(complete) > 50:
        results.append({"name": "DirectLiNGAM + deletion",
                         **_metrics(_lingam_baseline(complete))})
    imputed = all_data[VARS].copy()
    for c in imputed.columns:
        imputed[c] = imputed[c].fillna(imputed[c].mean())
    results.append({"name": "DirectLiNGAM + mean imputation",
                     **_metrics(_lingam_baseline(imputed))})
    results.append({"name": f"Our method (K={K})",
                     **_metrics(_build_graph(best["causal_orders"]))})
    with open(f"{output_dir}/baseline_comparison.json", "w") as f:
        json.dump(results, f, indent=2)
    for r in results:
        log(f"  [K={K}] {r['name']}: SHD={r['shd']}, F1={r['f1']:.3f}, edges={r['n_edges']}")


# ============================================================
# MASTER — RUN ALL K
# ============================================================

def run_all_K(K):
    log(f"\n{'='*70}")
    log(f"FED-CD PIPELINE — K = {K}")
    log(f"{'='*70}")

    cdir = f"data/clients_{K}"
    p2 = f"data/phase2_K{K}"
    p3 = f"data/phase3_K{K}"
    p3b = f"data/phase3b_K{K}"
    p4 = f"data/phase4_K{K}"
    p5 = f"data/phase5_K{K}"
    p6 = f"data/phase6_K{K}"

    log(f"\n--- Phase 1: Split into {K} clients ---")
    phase1_split(K, cdir)
    log(f"\n--- Phase 2: Classify + fit missingness ---")
    phase2_classify(K, cdir, p2)
    log(f"\n--- Phase 3: IPW correction ---")
    phase3_ipw(K, cdir, p2, p3)
    log(f"\n--- Phase 3B: Star graphs + cumulants ---")
    phase3b_cumulants(K, cdir, p3, p3b)
    phase3b_package(K, p3, p3b, p3b)
    log(f"\n--- Phase 4: Aggregate ---")
    phase4_aggregate(K, p3b, p4)
    log(f"\n--- Phase 5: Tune (F1-based) ---")
    best = phase5_tune(K, p4, p5)
    log(f"\n--- Phase 6: Save best + baselines ---")
    phase6_baselines(K, cdir, p5, p6, best)

    log(f"\n{'='*70}")
    log(f"K = {K} COMPLETE: SHD={best['shd']}, F1={best['f1']:.3f}, "
        f"P={best['precision']:.3f}, R={best['recall']:.3f}")
    log(f"{'='*70}")

    return {"K": K, **best}


def run_all_1_to_6():
    """Run all K = 1..6 sequentially."""
    all_results = []
    for K in [1, 2, 3, 4, 5, 6]:
        r = run_all_K(K)
        all_results.append(r)
    with open("data/all_K_results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    log(f"\n{'='*70}")
    log("FINAL SUMMARY")
    log(f"{'='*70}")
    log(f"{'K':>3} {'SHD':>5} {'F1':>6} {'P':>6} {'R':>6}")
    log("-" * 30)
    for r in all_results:
        log(f"{r['K']:>3} {r['shd']:>5} {r['f1']:>6.3f} "
            f"{r['precision']:>6.3f} {r['recall']:>6.3f}")
    return all_results



if __name__ == "__main__":
    run_all_1_to_6()


FED-CD PIPELINE — K = 1

--- Phase 1: Split into 1 clients ---
[K=1] Split 5400 rows into 1 clients
  Client 0: 5400 rows, 4.2% missing

--- Phase 2: Classify + fit missingness ---

--- Phase 3: IPW correction ---
  [K=1] Client 0: n_eff=4947.6/5400

--- Phase 3B: Star graphs + cumulants ---

--- Phase 4: Aggregate ---

--- Phase 5: Tune (F1-based) ---
  [K=1] Best (by F1): tau=0.5, c3=0.0, SHD=18, F1=0.526

--- Phase 6: Save best + baselines ---
  [K=1] DirectLiNGAM + deletion: SHD=56, F1=0.243, edges=55
  [K=1] DirectLiNGAM + mean imputation: SHD=50, F1=0.324, edges=55
  [K=1] Our method (K=1): SHD=18, F1=0.526, edges=19

K = 1 COMPLETE: SHD=18, F1=0.526, P=0.526, R=0.526

FED-CD PIPELINE — K = 2

--- Phase 1: Split into 2 clients ---
[K=2] Split 5400 rows into 2 clients
  Client 0: 2700 rows, 4.9% missing
  Client 1: 2700 rows, 3.4% missing

--- Phase 2: Classify + fit missingness ---

--- Phase 3: IPW correction ---
  [K=2] Client 0: n_eff=2697.4/2700
  [K=2] Client 1: n_eff=2496.

# 

# Asia Dataset
> asia_fed.py

In [18]:
"""
fed_cd_pipeline_asia.py — Federated Causal Discovery on ASIA dataset.
Asia is DISCRETE (binary) — uses LiM assumptions.

K matched to paper's alpha:
  K=1 -> alpha=1/8, K=2 -> alpha=2/8, K=3 -> alpha=3/8, K=4 -> alpha=4/8, K=5 -> alpha=5/8

All under data/asia_pipeline/ — no overlap with Sachs or synthetic.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import itertools


def log(msg):
    print(msg, flush=True)


# ============================================================
# PATHS — SEPARATE FROM SACHS AND SYNTHETIC
# ============================================================

ASIA_ROOT = "data/asia_pipeline"
ASIA_DATA = "data/asia.csv"
ASIA_GT = f"{ASIA_ROOT}/ground_truth_edges.txt"

# K matched to paper's alpha
CLIENT_COUNTS = [1, 2, 3, 4, 5]  # K=1 -> alpha=1/8, ..., K=5 -> alpha=5/8
MISSING_RATES = [0.1, 0.2, 0.3, 0.4, 0.5]
INT_WEIGHT = 2.0


# ============================================================
# ASIA GROUND TRUTH
# ============================================================

ASIA_TRUE_EDGES = [
    ("asia", "tub"),
    ("smoke", "lung"),
    ("smoke", "bronc"),
    ("tub", "either"),
    ("lung", "either"),
    ("either", "xray"),
    ("either", "dysp"),
    ("bronc", "dysp"),
]

ASIA_VARS = ["asia", "smoke", "tub", "lung", "bronc", "either", "xray", "dysp"]


def save_asia_gt():
    os.makedirs(ASIA_ROOT, exist_ok=True)
    with open(ASIA_GT, "w") as f:
        for u, v in ASIA_TRUE_EDGES:
            f.write(f"{u} -> {v}\n")
    log(f"[Asia] Ground truth: {len(ASIA_VARS)} nodes, {len(ASIA_TRUE_EDGES)} edges")
    log(f"[Asia] Saved to {ASIA_GT}")


def load_asia_gt():
    G = nx.DiGraph()
    with open(ASIA_GT) as f:
        for line in f:
            u, v = line.strip().split(" -> ")
            G.add_edge(u, v)
    return G, list(G.nodes()), list(G.edges())


# ============================================================
# GLOBALS
# ============================================================

VARS = []
TRUE_EDGES = []
GT_GRAPH = None


def _set_globals():
    global VARS, TRUE_EDGES, GT_GRAPH
    GT_GRAPH, VARS, TRUE_EDGES = load_asia_gt()


# ============================================================
# PHASE 1 — SPLIT + INJECT MISSINGNESS
# ============================================================

def _inject_var_missing(df, rate, seed):
    """Variable-level missingness: each variable gets `rate` fraction missing."""
    rng = np.random.RandomState(seed)
    out = df.copy().astype(float)
    for col in out.columns:
        mask = rng.rand(len(out)) < rate
        out.loc[mask, col] = np.nan
    return out


def phase1_split(K, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(ASIA_DATA)
    df = df[VARS].copy()

    rng = np.random.RandomState(42)
    idx = rng.permutation(len(df))
    splits = np.array_split(idx, K)

    log(f"[K={K}] Split {len(df)} rows into {K} clients "
        f"(missing rates cycle through {MISSING_RATES})")

    for k, split in enumerate(splits):
        cdf = df.iloc[split].reset_index(drop=True)
        rate = MISSING_RATES[k % len(MISSING_RATES)]
        cdf = _inject_var_missing(cdf, rate, seed=42 + k)
        cdf.to_csv(f"{output_dir}/client_{k}.csv", index=False)
        log(f"  Client {k}: {len(cdf)} rows, target={rate:.1f}, "
            f"actual={cdf.isna().mean().mean():.1%}")


# ============================================================
# PHASE 2 — CLASSIFY + FIT MISSINGNESS MODELS
# ============================================================

def _classify(df, cid):
    rates = df.isna().mean()
    V_o = [c for c in df.columns if rates[c] < 0.05]
    V_m = [c for c in df.columns if rates[c] >= 0.05]
    return {"client": cid, "V_o": V_o, "V_m": V_m,
            "missing_rates": {c: float(rates[c]) for c in df.columns},
            "n_rows": len(df)}


def _fit_model(df, target, parents, cid):
    R = df[target].isna().astype(int).values
    rate = R.mean()
    if rate < 0.05 or R.sum() < 10 or R.sum() == len(R):
        return None
    if len(parents) == 0:
        parents = [c for c in df.columns if c != target]
    pdata = df[parents]
    mask = pdata.notna().all(axis=1).values
    if mask.sum() < 50:
        return None
    X = pdata.values[mask].astype(float)
    R_obs = R[mask]
    if R_obs.sum() < 10 or R_obs.sum() == len(R_obs):
        return None
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    model = LogisticRegression(penalty="l2", C=1.0, max_iter=1000,
                                random_state=42, solver="lbfgs")
    model.fit(Xs, R_obs)
    probs = model.predict_proba(Xs)[:, 1]
    auc = roc_auc_score(R_obs, probs)
    return {"client": cid, "target": target, "parents": parents,
            "missing_rate": float(rate), "auc": float(auc),
            "model": model, "scaler": scaler}


def phase2_classify(K, client_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for k in range(K):
        df = pd.read_csv(f"{client_dir}/client_{k}.csv")
        cls = _classify(df, k)
        with open(f"{output_dir}/classification_{k}.json", "w") as f:
            json.dump(cls, f, indent=2)
        for target in cls["V_m"]:
            m = _fit_model(df, target, cls["V_o"], k)
            if m is None:
                continue
            with open(f"{output_dir}/model_{k}_{target}.pkl", "wb") as f:
                pickle.dump({"model": m["model"], "scaler": m["scaler"]}, f)
            meta = {kk: vv for kk, vv in m.items() if kk not in ["model", "scaler"]}
            with open(f"{output_dir}/meta_{k}_{target}.json", "w") as f:
                json.dump(meta, f, indent=2)


# ============================================================
# PHASE 3 — IPW CORRECTION
# ============================================================

def _weights_col(df, target, model_dict):
    model, scaler, parents = model_dict["model"], model_dict["scaler"], model_dict["parents"]
    obs = df[target].notna().values
    w = np.ones(len(df))
    if obs.sum() == 0:
        return w
    pdata = df.loc[obs, parents]
    if pdata.isna().any().any():
        pdata = pdata.fillna(pdata.mean())
    Xs = scaler.transform(pdata.values.astype(float))
    pm = np.clip(model.predict_proba(Xs)[:, 1], 1e-6, 1 - 1e-6)
    ww = np.minimum(1.0 / (1 - pm), np.percentile(1.0 / (1 - pm), 95))
    w[obs] = ww
    return w


def phase3_ipw(K, client_dir, phase2_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for k in range(K):
        df = pd.read_csv(f"{client_dir}/client_{k}.csv")
        with open(f"{phase2_dir}/classification_{k}.json") as f:
            cls = json.load(f)
        w = np.ones(len(df))
        for target in cls["V_m"]:
            path = f"{phase2_dir}/model_{k}_{target}.pkl"
            if not os.path.exists(path):
                continue
            with open(path, "rb") as f:
                md = pickle.load(f)
            with open(f"{phase2_dir}/meta_{k}_{target}.json") as f:
                md.update(json.load(f))
            w *= _weights_col(df, target, md)
        w = w / w.mean()
        n_eff = (w.sum() ** 2) / (w ** 2).sum()
        pd.DataFrame({"row": np.arange(len(df)), "weight": w}).to_csv(
            f"{output_dir}/weights_{k}.csv", index=False)
        with open(f"{output_dir}/ipw_meta_{k}.json", "w") as f:
            json.dump({"client": k, "n_rows": len(df), "n_eff": float(n_eff)}, f, indent=2)
        log(f"  [K={K}] Client {k}: n_eff={n_eff:.1f}/{len(df)}")


# ============================================================
# PHASE 3B — STAR GRAPHS + WEIGHTED CUMULANTS (BINARY)
# ============================================================

def _skeleton():
    G = nx.Graph()
    G.add_nodes_from(VARS)
    for u, v in TRUE_EDGES:
        G.add_edge(u, v)
    return G


def _star_graphs(skel):
    out = []
    for c in VARS:
        if c not in skel: continue
        periph = list(skel.neighbors(c))
        if periph:
            out.append({"center": c, "peripherals": periph})
    return out


def _c3(x, w):
    mu = np.sum(w * x) / np.sum(w)
    return np.sum(w * (x - mu) ** 3) / np.sum(w)


def _c12(x, y, w):
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) * (y - my) ** 2) / np.sum(w)


def _c21(x, y, w):
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) ** 2 * (y - my)) / np.sum(w)


def _pair_cums(df, w, x, y, min_obs=50):
    mask = df[x].notna() & df[y].notna()
    n = mask.sum()
    if n < min_obs: return None
    xv = df.loc[mask, x].values.astype(float)
    yv = df.loc[mask, y].values.astype(float)
    ww = w[mask] * (n / w[mask].sum())
    return {"C3_x": float(_c3(xv, ww)), "C3_y": float(_c3(yv, ww)),
            "C12_xy": float(_c12(xv, yv, ww)), "C21_xy": float(_c21(xv, yv, ww)),
            "C12_yx": float(_c12(yv, xv, ww)), "C21_yx": float(_c21(yv, xv, ww)),
            "n_obs": int(n)}


def phase3b_cumulants(K, client_dir, phase3_dir, output_dir):
    """Compute cumulants on Asia binary data."""
    os.makedirs(output_dir, exist_ok=True)
    sgs = _star_graphs(_skeleton())

    for k in range(K):
        df = pd.read_csv(f"{client_dir}/client_{k}.csv")
        w = pd.read_csv(f"{phase3_dir}/weights_{k}.csv")["weight"].values

        client_cum = {"client": k, "n_rows": len(df), "star_graphs": []}
        for sg in sgs:
            c, periph = sg["center"], sg["peripherals"]
            cums, nobs = {}, {}
            for p in periph:
                pc = _pair_cums(df, w, c, p)
                if pc is None:
                    continue
                cums[f"C3_{c}"] = pc["C3_x"]
                cums[f"C3_{p}"] = pc["C3_y"]
                cums[f"C12_{c}_{p}"] = pc["C12_xy"]
                cums[f"C21_{c}_{p}"] = pc["C21_xy"]
                cums[f"C12_{p}_{c}"] = pc["C12_yx"]
                cums[f"C21_{p}_{c}"] = pc["C21_yx"]
                nobs[f"{c}_{p}"] = pc["n_obs"]

            client_cum["star_graphs"].append({
                "center": c, "peripherals": periph,
                "cumulants": cums, "n_obs_pairs": nobs})
        with open(f"{output_dir}/cumulants_{k}.json", "w") as f:
            json.dump(client_cum, f, indent=2)


def phase3b_package(K, phase3_dir, phase3b_dir, output_dir):
    for k in range(K):
        with open(f"{phase3b_dir}/cumulants_{k}.json") as f:
            cum = json.load(f)
        with open(f"{phase3_dir}/ipw_meta_{k}.json") as f:
            ipw = json.load(f)
        with open(f"{output_dir}/client_cumulants_{k}.json", "w") as f:
            json.dump({"client": k, "n_rows": cum["n_rows"],
                        "n_eff": ipw["n_eff"],
                        "n_eff_ratio": ipw["n_eff"] / ipw["n_rows"],
                        "star_graphs": cum["star_graphs"]}, f, indent=2)


# ============================================================
# PHASE 4 — AGGREGATE
# ============================================================

def phase4_aggregate(K, phase3b_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    clients = []
    for k in range(K):
        with open(f"{phase3b_dir}/client_cumulants_{k}.json") as f:
            clients.append(json.load(f))
    total_neff = sum(c["n_eff"] for c in clients)
    by_center = {}
    for c in clients:
        for sg in c["star_graphs"]:
            by_center.setdefault(sg["center"], []).append((c, sg))
    agg = []
    for center, sg_list in by_center.items():
        periph = sg_list[0][1]["peripherals"]
        cums, nobs = {}, {}
        for c, sg in sg_list:
            wt = c["n_eff"] / total_neff
            for key, val in sg["cumulants"].items():
                cums.setdefault(key, 0.0)
                cums[key] += wt * val
            for pk, n in sg["n_obs_pairs"].items():
                nobs.setdefault(pk, 0)
                nobs[pk] += n
        agg.append({"center": center, "peripherals": periph,
                     "cumulants": cums, "n_obs_pairs": nobs})
    with open(f"{output_dir}/global_cumulants.json", "w") as f:
        json.dump({"n_clients": K, "total_n_eff": float(total_neff),
                    "star_graphs": agg}, f, indent=2)


# ============================================================
# PHASE 5 — TUNE (F1-BASED)
# ============================================================

TAU_RATIOS = [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
C3_THRESHOLDS = [0.0]


def _tau(cums, x, y):
    c3x = cums.get(f"C3_{x}", 0.0); c3y = cums.get(f"C3_{y}", 0.0)
    c12xy = cums.get(f"C12_{x}_{y}", 0.0); c21xy = cums.get(f"C21_{x}_{y}", 0.0)
    c12yx = cums.get(f"C12_{y}_{x}", 0.0); c21yx = cums.get(f"C21_{y}_{x}", 0.0)
    return (abs(abs(c3x) * c12xy - c21xy * c12yx),
            abs(abs(c3y) * c12yx - c21yx * c12xy))


def _find_source(cums, vars_, ratio):
    n = len(vars_)
    tau = {}
    for i in range(n):
        for j in range(n):
            if i == j: continue
            t, _ = _tau(cums, vars_[i], vars_[j])
            tau[(vars_[i], vars_[j])] = t
    if not tau: return None
    thr = ratio * max(tau.values())
    for s in vars_:
        if all(tau.get((s, j), 1.0) <= thr for j in vars_ if j != s):
            return s
    return None


def _eliminate(cums, src, desc):
    upd = dict(cums)
    c3s = cums.get(f"C3_{src}", 0.0)
    if abs(c3s) < 1e-10: return upd
    a = {i: np.clip(cums.get(f"C21_{i}_{src}", 0.0) / c3s, -2, 2)
         for i in desc if i != src}
    for i in desc:
        if i == src: continue
        upd[f"C3_{i}"] = cums.get(f"C3_{i}", 0.0) - (a.get(i, 0) ** 3) * c3s
    for i in desc:
        if i == src: continue
        for j in desc:
            if j == src or i == j: continue
            ai, aj = a.get(i, 0), a.get(j, 0)
            upd[f"C12_{i}_{j}"] = cums.get(f"C12_{i}_{j}", 0.0) - ai * (aj ** 2) * c3s
            upd[f"C21_{i}_{j}"] = cums.get(f"C21_{i}_{j}", 0.0) - (ai ** 2) * aj * c3s
    return upd


def _recover(cums, vars_, ratio):
    order, rem, cur = [], list(vars_), dict(cums)
    while len(rem) > 1:
        s = _find_source(cur, rem, ratio)
        if s is None: break
        order.append(s); rem.remove(s)
        if rem: cur = _eliminate(cur, s, rem)
    if len(rem) == 1: order.append(rem[0])
    return order


def _identifiable(sg, thr):
    return all(abs(sg["cumulants"].get(f"C3_{v}", 0.0)) >= thr
               for v in [sg["center"]] + sg["peripherals"])


def _build_graph(orders):
    G = nx.DiGraph()
    G.add_nodes_from(VARS)
    for o in orders:
        for i in range(len(o) - 1):
            G.add_edge(o[i], o[i + 1])
    return G


def _metrics(G):
    true = nx.DiGraph()
    true.add_nodes_from(VARS)
    true.add_edges_from(TRUE_EDGES)
    est = set(G.edges()); tru = set(true.edges())
    tp = len(est & tru); fp = len(est - tru); fn = len(tru - est)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"shd": len(tru - est) + len(est - tru), "f1": f1,
            "precision": p, "recall": r, "n_edges": len(est),
            "tp": tp, "fp": fp, "fn": fn}


def phase5_tune(K, phase4_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{phase4_dir}/global_cumulants.json") as f:
        gc = json.load(f)
    results = []
    for ratio, thr in itertools.product(TAU_RATIOS, C3_THRESHOLDS):
        orders = []
        for sg in gc["star_graphs"]:
            if not _identifiable(sg, thr): continue
            o = _recover(sg["cumulants"], [sg["center"]] + sg["peripherals"], ratio)
            if o: orders.append(o)
        m = _metrics(_build_graph(orders))
        results.append({"tau_ratio": ratio, "c3_threshold": thr,
                         "causal_orders": orders, **m})
    results.sort(key=lambda x: -x["f1"])
    best = results[0]
    with open(f"{output_dir}/tuning_results.json", "w") as f:
        json.dump(results, f, indent=2)
    log(f"  [K={K}] Best (by F1): tau={best['tau_ratio']}, c3={best['c3_threshold']}, "
        f"SHD={best['shd']}, F1={best['f1']:.3f}")
    return best


# ============================================================
# PHASE 6 — SAVE BEST + BASELINES
# ============================================================

BASELINE_PRUNE = 0.05


def _lingam_baseline(df):
    n = len(VARS)
    scores = {}
    for i in range(n):
        for j in range(i + 1, n):
            x, y = df[VARS[i]].values, df[VARS[j]].values
            r1 = y - LinearRegression().fit(x.reshape(-1, 1), y).predict(x.reshape(-1, 1))
            r2 = x - LinearRegression().fit(y.reshape(-1, 1), x).predict(y.reshape(-1, 1))
            scores[(VARS[i], VARS[j])] = abs(np.corrcoef(x, r1)[0, 1])
            scores[(VARS[j], VARS[i])] = abs(np.corrcoef(y, r2)[0, 1])
    G = nx.DiGraph()
    G.add_nodes_from(VARS)
    for i in range(n):
        for j in range(i + 1, n):
            u, v = VARS[i], VARS[j]
            suv, svu = scores.get((u, v), 1.0), scores.get((v, u), 1.0)
            if suv < BASELINE_PRUNE and suv < svu: G.add_edge(u, v)
            elif svu < BASELINE_PRUNE and svu < suv: G.add_edge(v, u)
    return G


def phase6_baselines(K, client_dir, phase5_dir, output_dir, best):
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/best_config.json", "w") as f:
        json.dump(best, f, indent=2)
    dfs = [pd.read_csv(f"{client_dir}/client_{k}.csv") for k in range(K)]
    all_data = pd.concat(dfs, ignore_index=True)
    results = []
    complete = all_data[VARS].dropna()
    if len(complete) > 50:
        results.append({"name": "DirectLiNGAM + deletion",
                         **_metrics(_lingam_baseline(complete))})
    imputed = all_data[VARS].copy()
    for c in imputed.columns:
        imputed[c] = imputed[c].fillna(imputed[c].mean())
    results.append({"name": "DirectLiNGAM + mean imputation",
                     **_metrics(_lingam_baseline(imputed))})
    results.append({"name": f"Our method (K={K})",
                     **_metrics(_build_graph(best["causal_orders"]))})
    with open(f"{output_dir}/baseline_comparison.json", "w") as f:
        json.dump(results, f, indent=2)
    for r in results:
        log(f"  [K={K}] {r['name']}: SHD={r['shd']}, F1={r['f1']:.3f}, "
            f"edges={r['n_edges']}")


# ============================================================
# MASTER — RUN ALL K
# ============================================================

def run_all_K(K):
    log(f"\n{'='*70}")
    log(f"ASIA PIPELINE — K = {K} (alpha = {K}/8)")
    log(f"{'='*70}")

    cdir = f"{ASIA_ROOT}/clients_{K}"
    p2 = f"{ASIA_ROOT}/phase2_K{K}"
    p3 = f"{ASIA_ROOT}/phase3_K{K}"
    p3b = f"{ASIA_ROOT}/phase3b_K{K}"
    p4 = f"{ASIA_ROOT}/phase4_K{K}"
    p5 = f"{ASIA_ROOT}/phase5_K{K}"
    p6 = f"{ASIA_ROOT}/phase6_K{K}"

    log(f"\n--- Phase 1: Split into {K} clients ---")
    phase1_split(K, cdir)
    log(f"\n--- Phase 2: Classify + fit missingness ---")
    phase2_classify(K, cdir, p2)
    log(f"\n--- Phase 3: IPW correction ---")
    phase3_ipw(K, cdir, p2, p3)
    log(f"\n--- Phase 3B: Star graphs + cumulants ---")
    phase3b_cumulants(K, cdir, p3, p3b)
    phase3b_package(K, p3, p3b, p3b)
    log(f"\n--- Phase 4: Aggregate ---")
    phase4_aggregate(K, p3b, p4)
    log(f"\n--- Phase 5: Tune (F1-based) ---")
    best = phase5_tune(K, p4, p5)
    log(f"\n--- Phase 6: Save best + baselines ---")
    phase6_baselines(K, cdir, p5, p6, best)

    log(f"\n{'='*70}")
    log(f"K = {K} (alpha={K}/8) COMPLETE: SHD={best['shd']}, F1={best['f1']:.3f}, "
        f"P={best['precision']:.3f}, R={best['recall']:.3f}")
    log(f"{'='*70}")
    return {"K": K, "alpha": f"{K}/8", **best}


def main():
    """Run Asia pipeline for K=1,2,3,4,5 (matched to alpha=1/8..5/8)."""
    log("="*70)
    log("ASIA PIPELINE — DISCRETE BINARY DATA")
    log("K matched to paper's alpha: K=1->1/8, K=2->2/8, ..., K=5->5/8")
    log("="*70)

    # 1. Save ground truth
    save_asia_gt()

    # 2. Set globals
    _set_globals()
    log(f"Variables: {VARS}")
    log(f"Ground truth edges: {len(TRUE_EDGES)}")

    # 3. Check data exists
    if not os.path.exists(ASIA_DATA):
        log(f"[ERROR] {ASIA_DATA} not found — run load_asia() first")
        return []

    # 4. Run all K = {1, 2, 3, 4, 5}
    all_results = []
    for K in CLIENT_COUNTS:
        r = run_all_K(K)
        all_results.append(r)

    # 5. Save summary
    with open(f"{ASIA_ROOT}/all_K_results.json", "w") as f:
        json.dump(all_results, f, indent=2)

    # 6. Print summary
    log(f"\n{'='*70}")
    log("FINAL SUMMARY — ASIA DATA")
    log(f"{'='*70}")
    log(f"{'K':>3} {'alpha':>6} {'SHD':>5} {'F1':>6} {'P':>6} {'R':>6}")
    log("-" * 40)
    for r in all_results:
        log(f"{r['K']:>3} {r['alpha']:>6} {r['shd']:>5} {r['f1']:>6.3f} "
            f"{r['precision']:>6.3f} {r['recall']:>6.3f}")

    return all_results


if __name__ == "__main__":
    main()

ASIA PIPELINE — DISCRETE BINARY DATA
K matched to paper's alpha: K=1->1/8, K=2->2/8, ..., K=5->5/8
[Asia] Ground truth: 8 nodes, 8 edges
[Asia] Saved to data/asia_pipeline/ground_truth_edges.txt
Variables: ['asia', 'tub', 'smoke', 'lung', 'bronc', 'either', 'xray', 'dysp']
Ground truth edges: 8

ASIA PIPELINE — K = 1 (alpha = 1/8)

--- Phase 1: Split into 1 clients ---
[K=1] Split 5000 rows into 1 clients (missing rates cycle through [0.1, 0.2, 0.3, 0.4, 0.5])
  Client 0: 5000 rows, target=0.1, actual=10.1%

--- Phase 2: Classify + fit missingness ---

--- Phase 3: IPW correction ---
  [K=1] Client 0: n_eff=4939.5/5000

--- Phase 3B: Star graphs + cumulants ---

--- Phase 4: Aggregate ---

--- Phase 5: Tune (F1-based) ---
  [K=1] Best (by F1): tau=0.7, c3=0.0, SHD=10, F1=0.583

--- Phase 6: Save best + baselines ---
  [K=1] DirectLiNGAM + deletion: SHD=30, F1=0.167, edges=28
  [K=1] DirectLiNGAM + mean imputation: SHD=22, F1=0.389, edges=28
  [K=1] Our method (K=1): SHD=10, F1=0.583, e